# Video Translation Pipeline - Debug Notebook

This notebook provides an interactive debugging environment for the YouTube video translation pipeline. Each cell represents a distinct stage that can be run and inspected independently.

## Pipeline Overview

The translation pipeline consists of these stages:

1. **Setup** - Load dependencies and configure API clients
2. **Input** - Specify video URL and translation settings
3. **Video Info** - Fetch metadata without downloading
4. **Download** - Download video with optional trimming
5. **Audio Extraction** - Extract audio track from video
6. **Audio Separation** - Separate vocals from background (Local Demucs)
7. **Speaker Diarization** - Identify different speakers (Local pyannote.audio)
8. **Voice Samples** - Extract samples for voice cloning
9. **Transcription** - Transcribe speech to text (OpenAI Whisper API)
10. **Translation** - Translate text to target language (Replicate Llama 3.1 405B)
11. **Speech Synthesis** - Generate translated speech (Local Chatterbox TTS)
12. **Audio Mixing** - Mix speech with background audio
13. **Subtitles** - Generate SRT subtitle file
14. **Final Merge** - Combine audio, video, and subtitles

## Model Execution

| Model | Execution | Notes |
|-------|-----------|-------|
| Demucs (audio separation) | **Local** | ~2GB VRAM |
| pyannote (speaker diarization) | **Local** | ~2GB VRAM, requires HuggingFace token |
| Chatterbox (TTS) | **Local** | ~4-6GB VRAM |
| Whisper | **Cloud** | OpenAI API |
| Llama 3.1 405B | **Cloud** | Replicate API (800GB VRAM required locally) |

## Requirements

- **Environment**: `.venv` with `requirements.txt` dependencies
- **GPU**: CUDA-capable GPU with 8GB+ VRAM recommended
- **API Keys**: 
  - `OPENAI_API_KEY` for Whisper transcription
  - `REPLICATE_API_TOKEN` for Llama translation
  - `HUGGINGFACE_TOKEN` for pyannote diarization models

## Usage

Run cells sequentially from top to bottom. Each cell checks prerequisites and provides detailed output for debugging.

## Pipeline Output Files Reference

Each cell checks if its output exists and skips if so.

| Cell | Step | Output File(s) |
|------|------|----------------|
| 4 | Video Download | `{video_id}.mp4` |
| 5 | Audio Extraction | `audio.wav` |
| 6 | Audio Separation | `vocals.wav`, `background.wav` |
| 7 | Speaker Diarization | `diarization.json` |
| 8 | Voice Samples | `voice_samples/` |
| 9 | Transcription | `transcription.json` |
| 10 | Translation | `translation.json` |
| 11 | Single-Speaker TTS | `translated_audio_single.wav` |
| 12 | Multi-Speaker TTS | `translated_audio_multi.wav` |
| 13 | Audio Mixing | `mixed_audio.wav` |
| 14 | Subtitles | `subtitles.srt` |
| 15 | Lip Sync | `lip_synced.mp4` |

**To re-run:** Delete the output file from the job folder.

## Cell 1: Setup and Configuration

This cell loads all required dependencies and initializes API clients for the pipeline.

### What This Cell Does

1. **Loads environment variables** from `.env` file
2. **Imports dependencies**:
   - Standard library (os, pathlib, subprocess, etc.)
   - Third-party (httpx, numpy, yt-dlp, deep-translator)
   - Local models (AudioSeparator, SpeakerDiarizer, ChatterboxMultilingualTTS)
   - Project modules (languages, config, utils)
3. **Configures API clients**:
   - OpenAI client for Whisper transcription
   - Replicate client for Llama 3.1 405B translation
4. **Sets up HuggingFace token** for pyannote diarization models

### Required Environment Variables

| Variable | Purpose | Required |
|----------|---------|----------|
| `OPENAI_API_KEY` | Whisper transcription API | Yes |
| `REPLICATE_API_TOKEN` | Llama 3.1 405B translation | Yes |
| `HUGGINGFACE_TOKEN` | pyannote speaker diarization | Yes |
| `OXYLABS_PROXY` | Optional proxy for yt-dlp | No |
| `YOUTUBE_COOKIES` | Optional cookies for age-restricted videos | No |

### Expected Output

Configuration status showing:
- Output directory path
- API key status (set/not set)
- HuggingFace token status
- Pipeline constants (sample rates, batch sizes, etc.)

In [ ]:
# =============================================================================
# Setup and Configuration
# =============================================================================

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

# =============================================================================
# RunPod Cloud Configuration
# =============================================================================
USE_RUNPOD_ENDPOINTS = True  # Set False to use local GPU models

# RunPod Endpoint IDs
RUNPOD_ENDPOINTS = {
    "demucs": "iawxgwernfm5s0",
    "whisper": "dcgrx4jdrpa2ml",
    "diarization": "r3qmbks8y3847n",
    "vllm": "700ucdghfohvrg",
    "chatterbox": "ia3chyjq7xkmn0",
    "musetalk": "ry3tcacm9dwt8w",
}

# Select GPU device (use GPU 1 for better performance) - must be before torch import
import os
RUNPOD_API_KEY = os.environ.get("RUNPOD_API_KEY")

# Only set CUDA device for local models
if not USE_RUNPOD_ENDPOINTS:
    os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch  # Import torch immediately after setting CUDA device
import base64

# Enable nested event loops for Jupyter compatibility
import nest_asyncio
nest_asyncio.apply()

# Standard library imports
import asyncio
import re
import subprocess
import tempfile
import time
from pathlib import Path
from typing import Optional

# Third-party imports
import httpx
import numpy as np
import replicate
from replicate.helpers import FileOutput
import yt_dlp
from deep_translator import GoogleTranslator
from openai import OpenAI

# Project imports - language mappings
from languages import (
    SUPPORTED_LANGUAGES,
    GOOGLE_LANG_CODES,
    ISO_639_1_TO_639_2,
    LANG_NAMES,
    get_language_name,
    get_google_code,
    get_iso_639_2_code,
)

# Project imports - configuration constants
from config import (
    GOOGLE_TRANSLATE_CHUNK_SIZE,
    LLM_MAX_SEGMENTS_PER_BATCH,
    LLM_BATCH_OVERLAP,
    LLM_MAX_TOKENS,
    LLM_TEMPERATURE,
    CHATTERBOX_SAMPLE_RATE,
    CHATTERBOX_CFG_WEIGHT,
    CHATTERBOX_EXAGGERATION,
    MIN_VOICE_SAMPLE_DURATION,
    MAX_VOICE_SAMPLE_DURATION,
    SIGNED_URL_EXPIRATION,
    REPLICATE_MODEL_LLAMA,
    REPLICATE_MODEL_CHATTERBOX,
    REPLICATE_MODEL_DEMUCS,
    REPLICATE_MODEL_DIARIZATION,
    BACKGROUND_VOLUME,
)

# Project imports - audio utilities
from utils.audio_helpers import read_wav_file, write_wav_file, resample_audio

# Local model imports - only load if not using RunPod
if not USE_RUNPOD_ENDPOINTS:
    from audio_processing import AudioSeparator, SpeakerDiarizer
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS

# =============================================================================
# RunPod Helper Function
# =============================================================================

async def call_runpod_endpoint(endpoint_key: str, input_data: dict, timeout: float = 600.0) -> dict:
    """
    Call a RunPod serverless endpoint and wait for result.
    
    Args:
        endpoint_key: Key in RUNPOD_ENDPOINTS dict (e.g., 'demucs', 'whisper')
        input_data: Input payload for the endpoint
        timeout: Request timeout in seconds (default 10 min)
    
    Returns:
        Output dict from the endpoint
    """
    endpoint_id = RUNPOD_ENDPOINTS[endpoint_key]
    base_url = f"https://api.runpod.ai/v2/{endpoint_id}"

    async with httpx.AsyncClient(timeout=timeout) as client:
        # Submit job
        response = await client.post(
            f"{base_url}/run",
            headers={"Authorization": f"Bearer {RUNPOD_API_KEY}"},
            json={"input": input_data},
        )
        response.raise_for_status()
        job_id = response.json()["id"]
        print(f"  Job submitted: {job_id}")

        # Poll for completion
        poll_count = 0
        while True:
            status_response = await client.get(
                f"{base_url}/status/{job_id}",
                headers={"Authorization": f"Bearer {RUNPOD_API_KEY}"},
            )
            status_response.raise_for_status()
            result = status_response.json()

            status = result.get("status")
            if status == "COMPLETED":
                return result.get("output", {})
            elif status == "FAILED":
                error = result.get("error", "Unknown error")
                raise RuntimeError(f"RunPod job failed: {error}")
            elif status in ("IN_QUEUE", "IN_PROGRESS"):
                poll_count += 1
                if poll_count % 15 == 0:  # Log every 30 seconds
                    print(f"  Status: {status} (polling...)")
                await asyncio.sleep(2.0)
            else:
                raise RuntimeError(f"Unknown status: {status}")


def run_runpod_sync(endpoint_key: str, input_data: dict, timeout: float = 600.0) -> dict:
    """Synchronous wrapper for call_runpod_endpoint."""
    return asyncio.get_event_loop().run_until_complete(
        call_runpod_endpoint(endpoint_key, input_data, timeout)
    )

# =============================================================================
# Helper Functions from cloud_translate.py
# =============================================================================

def parse_timestamp(ts: str) -> float:
    """
    Convert a timestamp string to float seconds.
    
    Handles format like "0:00:00.497812" (H:MM:SS.microseconds)
    or "1:30:45.5" (H:MM:SS.fraction).
    
    Args:
        ts: Timestamp string in format "H:MM:SS.fraction" or "H:MM:SS"
    
    Returns:
        Time in seconds as float (e.g., "1:30:45.5" -> 5445.5)
    """
    parts = ts.split(":")
    if len(parts) == 3:
        hours = int(parts[0])
        minutes = int(parts[1])
        seconds = float(parts[2])
        return hours * 3600 + minutes * 60 + seconds
    elif len(parts) == 2:
        minutes = int(parts[0])
        seconds = float(parts[1])
        return minutes * 60 + seconds
    else:
        return float(ts)


def format_timestamp_srt(seconds: float) -> str:
    """
    Convert seconds to SRT timestamp format (HH:MM:SS,mmm).
    
    Args:
        seconds: Time in seconds
    
    Returns:
        Timestamp string in SRT format
    """
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"


# =============================================================================
# Configuration from Environment
# =============================================================================

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
REPLICATE_API_TOKEN = os.getenv("REPLICATE_API_TOKEN")
HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE_TOKEN")

# Output directory
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "/tmp/yt-translate"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Proxy configuration (optional)
OXYLABS_PROXY = os.getenv("OXYLABS_PROXY")
YOUTUBE_COOKIES = os.getenv("YOUTUBE_COOKIES")

# =============================================================================
# Initialize API Clients
# =============================================================================

# OpenAI client for Whisper API (only needed if not using RunPod)
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY and not USE_RUNPOD_ENDPOINTS else None

# RunPod vLLM client (OpenAI-compatible)
if USE_RUNPOD_ENDPOINTS:
    vllm_client = OpenAI(
        api_key=RUNPOD_API_KEY,
        base_url=f"https://api.runpod.ai/v2/{RUNPOD_ENDPOINTS['vllm']}/openai/v1"
    )
else:
    vllm_client = None

# Replicate client is initialized via environment variable automatically
# It reads REPLICATE_API_TOKEN from the environment

# =============================================================================
# yt-dlp Configuration Helper
# =============================================================================

def get_ytdlp_opts(extra_opts: dict = None) -> dict:
    """Get yt-dlp options with optional proxy and cookie support."""
    opts = {'quiet': True, 'no_warnings': True}
    if OXYLABS_PROXY:
        opts['proxy'] = OXYLABS_PROXY
    if YOUTUBE_COOKIES:
        # Write cookies to temp file for yt-dlp
        cookie_file = OUTPUT_DIR / "youtube_cookies.txt"
        cookie_file.write_text(YOUTUBE_COOKIES)
        opts['cookiefile'] = str(cookie_file)
    if extra_opts:
        opts.update(extra_opts)
    return opts


# =============================================================================
# Display Configuration Status
# =============================================================================

print("=" * 60)
print("CONFIGURATION STATUS")
print("=" * 60)
print()
print(f"Mode: {'RunPod Cloud Endpoints' if USE_RUNPOD_ENDPOINTS else 'Local GPU Models'}")
print()
if USE_RUNPOD_ENDPOINTS:
    print("RunPod Endpoints:")
    for name, endpoint_id in RUNPOD_ENDPOINTS.items():
        print(f"  - {name}: {endpoint_id}")
    print(f"  - RUNPOD_API_KEY: {'Set (' + RUNPOD_API_KEY[:8] + '...)' if RUNPOD_API_KEY else 'NOT SET'}")
    print()
print(f"Output Directory: {OUTPUT_DIR}")
print(f"  - Exists: {OUTPUT_DIR.exists()}")
print()
print("API Keys:")
print(f"  - OPENAI_API_KEY: {'Set (' + OPENAI_API_KEY[:8] + '...)' if OPENAI_API_KEY else 'NOT SET'}")
print(f"  - REPLICATE_API_TOKEN: {'Set (' + REPLICATE_API_TOKEN[:8] + '...)' if REPLICATE_API_TOKEN else 'NOT SET'}")
print(f"  - HUGGINGFACE_TOKEN: {'Set (' + HUGGINGFACE_TOKEN[:8] + '...)' if HUGGINGFACE_TOKEN else 'NOT SET (needed for local diarization)'}")
print()
print("API Clients:")
if USE_RUNPOD_ENDPOINTS:
    print(f"  - RunPod vLLM Client: {'Initialized' if vllm_client else 'NOT INITIALIZED'}")
else:
    print(f"  - OpenAI Client: {'Initialized' if openai_client else 'NOT INITIALIZED (missing API key)'}")
    print(f"  - Replicate: {'Available' if REPLICATE_API_TOKEN else 'NOT AVAILABLE (missing API token)'}")
print()
print("Optional Configuration:")
print(f"  - Proxy (OXYLABS_PROXY): {'Configured' if OXYLABS_PROXY else 'Not configured'}")
print(f"  - YouTube Cookies: {'Configured' if YOUTUBE_COOKIES else 'Not configured'}")
print()
if not USE_RUNPOD_ENDPOINTS:
    print("Replicate Models:")
    print(f"  - LLM (Llama): {REPLICATE_MODEL_LLAMA}")
    print(f"  - TTS (Chatterbox): {REPLICATE_MODEL_CHATTERBOX.split(':')[0]}")
    print(f"  - Audio Separation (Demucs): {REPLICATE_MODEL_DEMUCS.split(':')[0]}")
    print(f"  - Speaker Diarization: {REPLICATE_MODEL_DIARIZATION.split(':')[0]}")
    print()
print("Pipeline Constants:")
print(f"  - Chatterbox Sample Rate: {CHATTERBOX_SAMPLE_RATE} Hz")
print(f"  - Min Voice Sample Duration: {MIN_VOICE_SAMPLE_DURATION} seconds")
print(f"  - Max Voice Sample Duration: {MAX_VOICE_SAMPLE_DURATION} seconds")
print(f"  - LLM Max Segments per Batch: {LLM_MAX_SEGMENTS_PER_BATCH}")
print(f"  - LLM Batch Overlap: {LLM_BATCH_OVERLAP}")
print(f"  - LLM Temperature: {LLM_TEMPERATURE}")
print()
print(f"Supported Languages: {len(SUPPORTED_LANGUAGES)}")
print(f"  {', '.join(SUPPORTED_LANGUAGES.values())}")
print()
print("=" * 60)
print("Setup complete! Proceed to the next cell to configure input.")
print("=" * 60)

## Cell 2: Input Configuration

This cell defines the input parameters for the pipeline. Modify these values to test different videos and configurations.

### Input Variables

| Variable | Description | Example |
|----------|-------------|--------|
| `VIDEO_URL` | URL of the video to translate | YouTube, Vimeo, Rumble, and 1700+ sites supported |
| `TARGET_LANGUAGE` | 2-letter ISO 639-1 language code | `"es"` (Spanish), `"ja"` (Japanese), `"fr"` (French) |
| `DURATION_LIMIT` | Max video length in seconds (None = full video) | `60` for 1 minute, `None` for no limit |
| `USE_MULTI_SPEAKER` | Enable per-speaker voice cloning | `True` for multiple speakers, `False` for single voice |

### Supported URL Formats

- **YouTube**: `https://www.youtube.com/watch?v=VIDEO_ID` or `https://youtu.be/VIDEO_ID`
- **Vimeo**: `https://vimeo.com/VIDEO_ID`
- **Rumble**: `https://rumble.com/VIDEO_ID.html`
- **Many more**: yt-dlp supports 1700+ sites ([full list](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md))

### Supported Language Codes

| Code | Language | Code | Language | Code | Language |
|------|----------|------|----------|------|----------|
| `ar` | Arabic | `he` | Hebrew | `no` | Norwegian |
| `zh` | Chinese | `hi` | Hindi | `pl` | Polish |
| `da` | Danish | `it` | Italian | `pt` | Portuguese |
| `nl` | Dutch | `ja` | Japanese | `ru` | Russian |
| `en` | English | `ko` | Korean | `es` | Spanish |
| `fi` | Finnish | `ms` | Malay | `sw` | Swahili |
| `fr` | French | `sv` | Swedish | `tr` | Turkish |
| `de` | German | `el` | Greek | | |

### Multi-Speaker Mode

- **`USE_MULTI_SPEAKER = True`**: Uses speaker diarization to identify different speakers, then clones each speaker's voice for synthesis. Best for interviews, podcasts, and multi-person content.
- **`USE_MULTI_SPEAKER = False`**: Uses a single voice for all speech synthesis. Faster and simpler, best for single-speaker content like tutorials or narration.

In [ ]:
# =============================================================================
# Input Configuration - Modify these values for your video
# =============================================================================

# Video URL to translate
# Supports: YouTube, Vimeo, Rumble, and 1700+ sites via yt-dlp
VIDEO_URL = "https://www.youtube.com/shorts/HHZ3CDsvjSg"  # Example: Rick Astley

# Target language (2-letter ISO 639-1 code)
# Examples: "es" (Spanish), "ja" (Japanese), "fr" (French), "de" (German)
TARGET_LANGUAGE = "ja"  # Spanish

# Duration limit in seconds (None = process full video)
# Useful for testing with long videos - set to 60 for first minute only
DURATION_LIMIT: int | None = 60  # Process only first 60 seconds

# Multi-speaker mode toggle
# True = Use speaker diarization and per-speaker voice cloning (better for interviews/podcasts)
# False = Single voice synthesis (faster, simpler, better for single-speaker content)
USE_MULTI_SPEAKER = True

# =============================================================================
# Display Configuration Summary
# =============================================================================

print("=" * 60)
print("INPUT CONFIGURATION")
print("=" * 60)
print()
print(f"Video URL: {VIDEO_URL}")
print(f"Target Language: {TARGET_LANGUAGE} ({get_language_name(TARGET_LANGUAGE)})")
print(f"Duration Limit: {DURATION_LIMIT} seconds" if DURATION_LIMIT else "Duration Limit: None (full video)")
print(f"Multi-Speaker Mode: {'Enabled' if USE_MULTI_SPEAKER else 'Disabled'}")
print()

# Validate target language
if TARGET_LANGUAGE not in SUPPORTED_LANGUAGES:
    print(f"WARNING: '{TARGET_LANGUAGE}' is not in supported languages!")
    print(f"   Supported: {list(SUPPORTED_LANGUAGES.keys())}")
else:
    print(f"Target language '{TARGET_LANGUAGE}' is supported")

print()
print("=" * 60)
print("Configuration complete! Proceed to video info retrieval.")
print("=" * 60)

## Cell 3: Video Info Retrieval

This cell fetches video metadata **without downloading** the actual video file.

### How it works

yt-dlp's `extract_info()` with `download=False` makes an API call to the video platform to retrieve metadata:

- **Title**: Video title as shown on the platform
- **Duration**: Length in seconds (needed for price calculation)
- **Video ID**: Platform-specific identifier (e.g., YouTube's 11-character ID)
- **Thumbnail URL**: Preview image URL

### Price Calculation

The estimated price is calculated using the formula:

```
price = ceil(duration_seconds / 60) * PRICE_PER_MINUTE
```

- Rounded up to the nearest minute
- Minimum charge is 1 minute
- Default rate: $0.50 per minute (configurable via `PRICE_PER_MINUTE_CENTS`)

### Error Handling

Common errors include:
- **Invalid URL**: Unsupported platform or malformed URL
- **Private/Restricted**: Video requires authentication or is geo-blocked
- **Removed/Unavailable**: Video no longer exists

In [ ]:
# =============================================================================
# Video Info Retrieval - Fetch metadata without downloading
# =============================================================================

# Price per minute in cents (configurable via environment variable)
PRICE_PER_MINUTE_CENTS = int(os.getenv("PRICE_PER_MINUTE_CENTS", "50"))  # $0.50 default


def calculate_price_cents(duration_seconds: int) -> int:
    """
    Calculate price in cents based on video duration.
    
    Price is calculated per minute, rounded up.
    Minimum price is 1 minute.
    """
    minutes = max(1, (duration_seconds + 59) // 60)  # Round up to nearest minute
    return minutes * PRICE_PER_MINUTE_CENTS


def format_duration(seconds: int) -> str:
    """Format duration in seconds to human-readable string (HH:MM:SS or MM:SS)."""
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    if hours > 0:
        return f"{hours}:{minutes:02d}:{secs:02d}"
    return f"{minutes}:{secs:02d}"


# =============================================================================
# Extract Video Info
# =============================================================================

video_info = None  # Will hold the extracted info for later cells

print("=" * 60)
print("VIDEO INFO RETRIEVAL")
print("=" * 60)
print()
print(f"Fetching info for: {VIDEO_URL}")
print()

try:
    with yt_dlp.YoutubeDL(get_ytdlp_opts()) as ydl:
        info = ydl.extract_info(VIDEO_URL, download=False)
        
        if info is None:
            raise ValueError("Could not extract video information from the provided URL")
        
        # Extract metadata
        title = info.get('title', 'Unknown Title')
        duration = info.get('duration')
        video_id = info.get('id', 'unknown')
        
        if duration is None:
            raise ValueError("Could not determine video duration. The URL may not point to a valid video.")
        
        # Get thumbnail (yt-dlp provides various thumbnail options)
        thumbnail = info.get('thumbnail')
        if not thumbnail:
            thumbnails = info.get('thumbnails', [])
            if thumbnails:
                thumbnail = thumbnails[-1].get('url')
        
        # Calculate price
        price_cents = calculate_price_cents(duration)
        
        # Store for later cells
        video_info = {
            'title': title,
            'duration': duration,
            'video_id': video_id,
            'thumbnail': thumbnail,
            'price_cents': price_cents,
            'uploader': info.get('uploader', 'Unknown'),
            'extractor': info.get('extractor', 'Unknown'),
        }
        
        # Display results
        print("Video Metadata")
        print("-" * 40)
        print(f"Title:      {title}")
        print(f"Video ID:   {video_id}")
        print(f"Duration:   {format_duration(duration)} ({duration} seconds)")
        print(f"Uploader:   {video_info['uploader']}")
        print(f"Platform:   {video_info['extractor']}")
        print()
        print(f"Thumbnail URL:")
        print(f"  {thumbnail}")
        print()
        print("Price Estimation")
        print("-" * 40)
        minutes = max(1, (duration + 59) // 60)
        print(f"Billable minutes: {minutes} (rounded up from {duration/60:.1f} min)")
        print(f"Rate: ${PRICE_PER_MINUTE_CENTS/100:.2f} per minute")
        print(f"Estimated price: ${price_cents/100:.2f} ({price_cents} cents)")
        print()
        
        # Note about duration limit
        if DURATION_LIMIT and duration > DURATION_LIMIT:
            limited_price_cents = calculate_price_cents(DURATION_LIMIT)
            print(f"With DURATION_LIMIT={DURATION_LIMIT}s: ${limited_price_cents/100:.2f}")
            print(f"  (Processing only first {format_duration(DURATION_LIMIT)} of {format_duration(duration)})")
        
        print()
        print("=" * 60)
        print("Video info retrieved successfully! Proceed to download.")
        print("=" * 60)

except yt_dlp.utils.DownloadError as e:
    print(f"ERROR: Invalid video URL or video not accessible")
    print(f"  Details: {str(e)}")
    print()
    print("Common causes:")
    print("  - URL is malformed or unsupported")
    print("  - Video is private or age-restricted")
    print("  - Video has been removed")
    print("  - Geographic restrictions apply")
    print()
    print("Try: Double-check the URL or use YouTube cookies for restricted videos.")

except Exception as e:
    print(f"ERROR: Failed to extract video information")
    print(f"  Exception type: {type(e).__name__}")
    print(f"  Details: {str(e)}")

## Cell 4: Video Download

This cell downloads the video file using yt-dlp and optionally trims it if `DURATION_LIMIT` is set.

### Download Process

1. **Format Selection**: yt-dlp selects `bestvideo+bestaudio/best` - the highest quality video and audio streams
2. **Merge Output**: Streams are merged into a single MP4 container using ffmpeg
3. **Duration Trimming**: If `DURATION_LIMIT` is set, ffmpeg trims the video using stream copy (fast, no re-encoding)

### Format Selection Details

The `bestvideo+bestaudio/best` format string means:
- Try to get the best video and best audio as separate streams, then merge them
- If that fails (e.g., audio-only or video-only), fall back to the single best stream

### Duration Trimming

When `DURATION_LIMIT` is set:
```bash
ffmpeg -i input.mp4 -t {DURATION_LIMIT} -c:v copy -c:a copy -y output.mp4
```

- `-t {DURATION_LIMIT}`: Stop writing output after this many seconds
- `-c:v copy -c:a copy`: Stream copy (no re-encoding) - very fast
- The original file is replaced with the trimmed version

### Output

- **video_path**: Path to the downloaded (and optionally trimmed) video file
- **File size**: Displayed in human-readable format (MB/GB)

In [ ]:
# =============================================================================
# Video Download - Download and optionally trim the video
# =============================================================================

def format_file_size(size_bytes: int) -> str:
    """Format file size in human-readable format."""
    if size_bytes < 1024:
        return f"{size_bytes} B"
    elif size_bytes < 1024 * 1024:
        return f"{size_bytes / 1024:.1f} KB"
    elif size_bytes < 1024 * 1024 * 1024:
        return f"{size_bytes / (1024 * 1024):.1f} MB"
    else:
        return f"{size_bytes / (1024 * 1024 * 1024):.2f} GB"


# =============================================================================
# Download Video
# =============================================================================

video_path = None  # Will hold the path to the downloaded video

print("=" * 60)
print("VIDEO DOWNLOAD")
print("=" * 60)
print()

# Check prerequisite from previous cell
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
else:
    # === Skip check ===
    _skip = False
    _job_dir = OUTPUT_DIR / video_info.get("video_id", "")
    if (_job_dir / f"{video_info.get('video_id', '')}.mp4").exists():
        print(f"SKIPPING: Video already exists")
        video_path = _job_dir / f"{video_info.get('video_id', '')}.mp4"
        video_info["video_path"] = video_path
        video_info["actual_duration"] = min(video_info.get("duration", 0), DURATION_LIMIT) if DURATION_LIMIT else video_info.get("duration", 0)
        _skip = True

    if not _skip:
        try:
            video_id = video_info['video_id']
            title = video_info['title']
            duration = video_info['duration']
        
            # Create job directory for this video
            job_dir = OUTPUT_DIR / video_id
            job_dir.mkdir(parents=True, exist_ok=True)
        
            print(f"Downloading: {title}")
            print(f"Video ID: {video_id}")
            print(f"Output directory: {job_dir}")
            print()
        
            # Configure yt-dlp for download
            ydl_opts = get_ytdlp_opts({
                'format': 'bestvideo+bestaudio/best',
                'outtmpl': str(job_dir / f"{video_id}.%(ext)s"),
                'merge_output_format': 'mp4',
                'quiet': False,  # Show progress for debugging
                'no_warnings': False,
            })
        
            # Download the video
            print("Downloading video...")
            print("-" * 40)
            download_start = time.time()
        
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([VIDEO_URL])
        
            download_time = time.time() - download_start
            print("-" * 40)
            print(f"Download completed in {download_time:.1f} seconds")
            print()
        
            # Find the downloaded video file
            video_path = None
            for ext in ['mp4', 'mkv', 'webm']:
                p = job_dir / f"{video_id}.{ext}"
                if p.exists():
                    video_path = p
                    break
        
            if video_path is None:
                raise FileNotFoundError(f"Could not find downloaded video in {job_dir}")
        
            original_size = video_path.stat().st_size
            print(f"Downloaded file: {video_path.name}")
            print(f"Original size: {format_file_size(original_size)}")
            print()
        
            # Apply duration limit if specified
            if DURATION_LIMIT is not None and duration > DURATION_LIMIT:
                print(f"Applying duration limit: {DURATION_LIMIT} seconds")
                print(f"  (Trimming from {format_duration(duration)} to {format_duration(DURATION_LIMIT)})")
                print()
            
                trimmed_path = job_dir / f"{video_id}_trimmed.mp4"
            
                # Trim using ffmpeg with stream copy (fast, no re-encoding)
                trim_cmd = [
                    'ffmpeg', '-i', str(video_path),
                    '-t', str(DURATION_LIMIT),
                    '-c:v', 'copy', '-c:a', 'copy',
                    '-y', str(trimmed_path)
                ]
            
                print("Running ffmpeg trim...")
                trim_start = time.time()
                result = subprocess.run(trim_cmd, capture_output=True, text=True)
                trim_time = time.time() - trim_start
            
                if result.returncode != 0:
                    print(f"ERROR: ffmpeg trim failed")
                    print(f"  stderr: {result.stderr}")
                else:
                    # Replace original with trimmed version
                    video_path.unlink()  # Delete original
                    # Rename trimmed to standard name
                    final_path = job_dir / f"{video_id}.mp4"
                    trimmed_path.rename(final_path)
                    video_path = final_path
                
                    trimmed_size = video_path.stat().st_size
                    print(f"Trim completed in {trim_time:.1f} seconds")
                    print(f"Trimmed size: {format_file_size(trimmed_size)}")
                    print(f"Size reduction: {(1 - trimmed_size/original_size)*100:.1f}%")
        
            print()
            print("Download Summary")
            print("-" * 40)
            print(f"Video path: {video_path}")
            print(f"Final size: {format_file_size(video_path.stat().st_size)}")
        
            # Update video_info with actual processed duration
            actual_duration = min(duration, DURATION_LIMIT) if DURATION_LIMIT else duration
            video_info['actual_duration'] = actual_duration
            video_info['video_path'] = video_path
            print(f"Actual duration: {format_duration(actual_duration)} ({actual_duration} seconds)")
        
            print()
            print("=" * 60)
            print("Video downloaded successfully! Proceed to audio extraction.")
            print("=" * 60)
        
        except subprocess.CalledProcessError as e:
            print(f"ERROR: ffmpeg command failed")
            print(f"  Command: {' '.join(e.cmd)}")
            print(f"  Return code: {e.returncode}")
            if e.stderr:
                print(f"  stderr: {e.stderr}")
    
        except yt_dlp.utils.DownloadError as e:
            print(f"ERROR: Video download failed")
            print(f"  Details: {str(e)}")
            print()
            print("Common causes:")
            print("  - Network connectivity issues")
            print("  - Video is age-restricted (try using cookies)")
            print("  - Rate limiting (try using a proxy)")
            print("  - DRM-protected content")
    
        except Exception as e:
            print(f"ERROR: Unexpected error during download")
            print(f"  Exception type: {type(e).__name__}")
            print(f"  Details: {str(e)}")

## Cell 5: Audio Extraction

This cell extracts audio from the downloaded video file for processing by the speech-to-text pipeline.

### Why 16kHz Mono PCM?

OpenAI's Whisper API works best with audio that matches its training data format:

- **16kHz Sample Rate**: Whisper was trained on 16kHz audio. Higher rates waste bandwidth without improving accuracy.
- **Mono (1 channel)**: Speech recognition doesn't benefit from stereo. Combining channels reduces file size by 50%.
- **PCM 16-bit**: Uncompressed format with no quality loss. The `.wav` extension signals this to downstream tools.

### ffmpeg Command Breakdown

```bash
ffmpeg -i video.mp4 -vn -acodec pcm_s16le -ar 16000 -ac 1 -y audio.wav
```

| Flag | Description |
|------|-------------|
| `-i video.mp4` | Input video file |
| `-vn` | No video output (audio only) |
| `-acodec pcm_s16le` | PCM 16-bit signed little-endian codec |
| `-ar 16000` | Set audio sample rate to 16kHz |
| `-ac 1` | Convert to mono (1 audio channel) |
| `-y` | Overwrite output file if exists |

### Output

- **audio_path**: Path to the extracted `.wav` file
- **Audio duration**: Should match video duration (or `DURATION_LIMIT` if set)
- **File size**: Approximately `duration_seconds * 32000` bytes (16000 Hz × 2 bytes × mono)

In [ ]:
# =============================================================================
# Audio Extraction - Extract audio from video for transcription
# =============================================================================

# Audio extraction parameters
WHISPER_SAMPLE_RATE = 16000  # 16kHz for Whisper API


def get_audio_duration(audio_path: Path) -> float:
    """
    Get the duration of an audio file using ffprobe.
    
    Args:
        audio_path: Path to the audio file
    
    Returns:
        Duration in seconds
    """
    cmd = [
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1',
        str(audio_path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0 and result.stdout.strip():
        return float(result.stdout.strip())
    return 0.0


# =============================================================================
# Extract Audio from Video
# =============================================================================

audio_path = None  # Will hold the path to the extracted audio

print("=" * 60)
print("AUDIO EXTRACTION")
print("=" * 60)
print()

# Check prerequisites from previous cells
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('video_path') is None:
    print("ERROR: video_path not available. Run the Video Download cell first.")
else:
    # === Skip check ===
    _skip = False
    _job_dir = video_info["video_path"].parent
    if (_job_dir / "audio.wav").exists():
        print(f"SKIPPING: Audio already exists")
        audio_path = _job_dir / "audio.wav"
        video_info["audio_path"] = audio_path
        _skip = True

    if not _skip:
        try:
            video_path = video_info['video_path']
            video_id = video_info['video_id']
            job_dir = video_path.parent
        
            # Define output audio path
            audio_path = job_dir / "audio.wav"
        
            print(f"Source video: {video_path}")
            print(f"Output audio: {audio_path}")
            print(f"Target format: {WHISPER_SAMPLE_RATE} Hz, mono, PCM 16-bit")
            print()
        
            # Build ffmpeg command for audio extraction
            ffmpeg_cmd = [
                'ffmpeg',
                '-i', str(video_path),        # Input video
                '-vn',                         # No video output
                '-acodec', 'pcm_s16le',       # PCM 16-bit signed little-endian
                '-ar', str(WHISPER_SAMPLE_RATE),  # 16kHz sample rate
                '-ac', '1',                    # Mono (1 channel)
                '-y',                          # Overwrite output file
                str(audio_path)
            ]
        
            print("Running ffmpeg...")
            print(f"  Command: {' '.join(ffmpeg_cmd)}")
            print()
        
            extract_start = time.time()
            result = subprocess.run(ffmpeg_cmd, capture_output=True, text=True)
            extract_time = time.time() - extract_start
        
            if result.returncode != 0:
                print(f"ERROR: ffmpeg audio extraction failed")
                print(f"  Return code: {result.returncode}")
                print(f"  stderr: {result.stderr}")
                audio_path = None
            else:
                # Verify the output file exists
                if not audio_path.exists():
                    raise FileNotFoundError(f"Audio extraction failed: {audio_path} not created")
            
                # Get file info
                audio_size = audio_path.stat().st_size
                audio_duration = get_audio_duration(audio_path)
            
                # Store in video_info for later cells
                video_info['audio_path'] = audio_path
                video_info['audio_duration'] = audio_duration
            
                # Display results
                print("Audio Extraction Summary")
                print("-" * 40)
                print(f"Audio path: {audio_path}")
                print(f"File size: {format_file_size(audio_size)}")
                print(f"Duration: {format_duration(int(audio_duration))} ({audio_duration:.2f} seconds)")
                print(f"Sample rate: {WHISPER_SAMPLE_RATE} Hz")
                print(f"Channels: 1 (mono)")
                print(f"Bit depth: 16-bit")
                print(f"Extraction time: {extract_time:.2f} seconds")
                print()
            
                # Calculate expected file size and compare
                expected_size = int(audio_duration * WHISPER_SAMPLE_RATE * 2)  # 2 bytes per sample
                expected_size += 44  # WAV header
                print(f"Expected size (approx): {format_file_size(expected_size)}")
                print(f"Actual size: {format_file_size(audio_size)}")
            
                print()
                print("=" * 60)
                print("Audio extracted successfully! Proceed to audio separation.")
                print("=" * 60)
    
        except subprocess.CalledProcessError as e:
            print(f"ERROR: ffmpeg command failed")
            print(f"  Return code: {e.returncode}")
            if e.stderr:
                print(f"  stderr: {e.stderr}")
    
        except FileNotFoundError as e:
            print(f"ERROR: {str(e)}")
    
        except Exception as e:
            print(f"ERROR: Unexpected error during audio extraction")
            print(f"  Exception type: {type(e).__name__}")
            print(f"  Details: {str(e)}")

## Cell 6: Audio Separation

This cell separates vocals from background music using **local Demucs** model. This is essential for:
- Clean transcription (vocals only)
- Background audio preservation for final mix
- Voice sample extraction for cloning

### What This Cell Does

1. **Loads Demucs model** locally (htdemucs)
2. **Separates audio** into vocals and background tracks
3. **Saves both tracks** as separate WAV files

### Model Details

| Property | Value |
|----------|-------|
| Model | htdemucs (local) |
| Execution | Local GPU/CPU |
| VRAM Usage | ~2GB |
| Output Sample Rate | 44.1kHz |

### Expected Output

- `vocals.wav` - Isolated vocal track for transcription
- `background.wav` - Instrumental/background track for mixing
- Timing information and file sizes

In [ ]:
# =============================================================================
# Audio Separation - Demucs (GPU accelerated)
# =============================================================================

import soundfile as sf
if not USE_RUNPOD_ENDPOINTS:
    from audio_processing import AudioSeparator

# =============================================================================
# Audio Separation
# =============================================================================

vocals_path = None  # Will hold the path to isolated vocals
background_path = None  # Will hold the path to background audio

print("=" * 60)
print(f"AUDIO SEPARATION ({'RunPod Demucs' if USE_RUNPOD_ENDPOINTS else 'Local Demucs'})")
print("=" * 60)
print()

# Check prerequisites from previous cells
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('audio_path') is None:
    print("ERROR: audio_path not available. Run the Audio Extraction cell first.")
else:
    # === Skip check ===
    _skip = False
    _job_dir = video_info["audio_path"].parent
    if (_job_dir / "vocals.wav").exists() and (_job_dir / "background.wav").exists():
        print(f"SKIPPING: Separated audio already exists")
        vocals_path = _job_dir / "vocals.wav"
        background_path = _job_dir / "background.wav"
        video_info["vocals_path"] = vocals_path
        video_info["background_path"] = background_path
        _skip = True

    if not _skip:
        try:
            audio_path = video_info['audio_path']
            job_dir = audio_path.parent
        
            print(f"Source audio: {audio_path}")
            print()
        
            # Define output paths
            vocals_path = job_dir / "vocals.wav"
            background_path = job_dir / "background.wav"
        
            separation_start = time.time()
            
            if USE_RUNPOD_ENDPOINTS:
                # =====================================================
                # RunPod Demucs Endpoint
                # =====================================================
                print("Using RunPod Demucs endpoint...")
                print()
                
                # Read audio and encode to base64
                with open(audio_path, "rb") as f:
                    audio_base64 = base64.b64encode(f.read()).decode("utf-8")
                print(f"  Audio encoded: {len(audio_base64) / 1e6:.1f} MB base64")
                
                # Call RunPod endpoint
                result = run_runpod_sync("demucs", {"audio_base64": audio_base64})
                
                if result.get("status") != "success":
                    raise RuntimeError(f"Demucs failed: {result.get('error', 'Unknown error')}")
                
                # Download audio files from RunPod storage URLs
                print("  Downloading vocals from RunPod storage...")
                vocals_response = httpx.get(result["vocals_url"], timeout=300.0)
                vocals_data = vocals_response.content
                
                print("  Downloading background from RunPod storage...")
                background_response = httpx.get(result["background_url"], timeout=300.0)
                background_data = background_response.content
                
                with open(vocals_path, "wb") as f:
                    f.write(vocals_data)
                print(f"  Vocals saved: {len(vocals_data) / 1e6:.1f} MB")
                
                with open(background_path, "wb") as f:
                    f.write(background_data)
                print(f"  Background saved: {len(background_data) / 1e6:.1f} MB")
                
                # Get metrics
                metrics = result.get("metrics", {})
                print(f"  Inference time: {metrics.get('separation_seconds', 'N/A')}s")
                
            else:
                # =====================================================
                # Local Demucs (GPU)
                # =====================================================
                print("Using local Demucs model (htdemucs)")
                
                # Initialize separator (auto-detects GPU/CPU)
                separator = AudioSeparator()
                print(f"Device: {separator.device}")
                print()
                
                # Perform separation
                print("Separating audio into vocals and background...")
                print("-" * 40)
                
                vocals_array, background_array, sample_rate = separator.separate(audio_path)
                
                print("-" * 40)
                print()
                
                # Save separated audio
                print("Saving separated stems...")
                print(f"  - Saving vocals to: {vocals_path}")
                sf.write(vocals_path, vocals_array, sample_rate)
                
                print(f"  - Saving background to: {background_path}")
                sf.write(background_path, background_array, sample_rate)
            
            separation_time = time.time() - separation_start
            print()
        
            # Get file info
            vocals_size = vocals_path.stat().st_size
            background_size = background_path.stat().st_size
        
            # Store in video_info for later cells
            video_info['vocals_path'] = vocals_path
            video_info['background_path'] = background_path
        
            # Display results
            print("Audio Separation Summary")
            print("-" * 40)
            print(f"Vocals path: {vocals_path}")
            print(f"  - Size: {format_file_size(vocals_size)}")
            print()
            print(f"Background path: {background_path}")
            print(f"  - Size: {format_file_size(background_size)}")
            print()
            print("Timing")
            print("-" * 40)
            print(f"Separation: {separation_time:.1f} seconds")
        
            print()
            print("=" * 60)
            print("Audio separated successfully! Proceed to speaker diarization.")
            print("=" * 60)
    
        except Exception as e:
            print(f"ERROR: Unexpected error during audio separation")
            print(f"  Exception type: {type(e).__name__}")
            print(f"  Details: {str(e)}")
            print()
            print("Fallback: Using original audio as vocals (no separation)")
            vocals_path = audio_path
            background_path = audio_path
            video_info['vocals_path'] = vocals_path
            video_info['background_path'] = background_path


## Cell 7: Speaker Diarization

This cell identifies different speakers in the audio using **local pyannote.audio** pipeline. This is essential for multi-speaker content like interviews, podcasts, or panel discussions.

### What is Speaker Diarization?

Speaker diarization answers the question "who spoke when?" by segmenting audio into time ranges and assigning speaker labels. The model:

1. **Detects speech segments** - Identifies when speech occurs
2. **Clusters speakers** - Groups similar voice characteristics together
3. **Assigns labels** - Tags each segment with a speaker ID (SPEAKER_00, SPEAKER_01, etc.)

### Model Details

| Property | Value |
|----------|-------|
| Model | pyannote/speaker-diarization-3.1 (local) |
| Execution | Local GPU/CPU |
| VRAM Usage | ~2GB |
| Requires | `HUGGINGFACE_TOKEN` environment variable |

### Prerequisites

- Vocals audio from Audio Separation cell
- Valid HuggingFace token with pyannote model access

### Expected Output

- Number of speakers detected
- Per-speaker statistics (segment count, total duration)
- Detailed segment table with timestamps

### Notes

- First run downloads model weights (~1GB)
- GPU significantly speeds up processing
- Results stored in `video_info['diarization_segments']`

In [ ]:
# =============================================================================
# Speaker Diarization - Identify different speakers in the audio
# =============================================================================

import json as json_module  # Avoid conflict with potential variable names

# =============================================================================
# Speaker Diarization
# =============================================================================

diarization_segments = []  # Will hold the parsed diarization segments

print("=" * 60)
print(f"SPEAKER DIARIZATION ({'RunPod' if USE_RUNPOD_ENDPOINTS else 'Local pyannote.audio'})")
print("=" * 60)
print()

# Check prerequisites from previous cells
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('vocals_path') is None:
    print("ERROR: vocals_path not available. Run the Audio Separation cell first.")
elif not USE_RUNPOD_ENDPOINTS and not HUGGINGFACE_TOKEN:
    print("ERROR: HUGGINGFACE_TOKEN not set. Required for pyannote.audio diarization.")
    print("  Set the environment variable and re-run the Setup cell.")
else:
    # === Skip check ===
    _skip = False
    _job_dir = video_info["vocals_path"].parent
    if (_job_dir / "diarization.json").exists():
        print(f"SKIPPING: Diarization already exists")
        import json as _jm
        with open(_job_dir / "diarization.json") as _f:
            diarization_segments = _jm.load(_f)
        video_info["diarization_segments"] = diarization_segments
        video_info["num_speakers"] = len(set(s.get("speaker", "") for s in diarization_segments))
        _skip = True

    if not _skip:
        try:
            vocals_path = video_info['vocals_path']
            job_dir = vocals_path.parent
        
            print(f"Source audio: {vocals_path}")
            print()
        
            api_start = time.time()
            
            if USE_RUNPOD_ENDPOINTS:
                # =====================================================
                # RunPod Diarization Endpoint
                # =====================================================
                print("Using RunPod pyannote diarization endpoint...")
                print()
                
                # Read audio and encode to base64
                with open(vocals_path, "rb") as f:
                    audio_base64 = base64.b64encode(f.read()).decode("utf-8")
                print(f"  Audio encoded: {len(audio_base64) / 1e6:.1f} MB base64")
                
                # Call RunPod endpoint
                result = run_runpod_sync("diarization", {"audio_base64": audio_base64})
                
                if result.get("status") == "error":
                    raise RuntimeError(f"Diarization failed: {result.get('error', 'Unknown error')}")
                
                # Extract segments from result
                segments = result.get("segments", result.get("diarization", []))
                
            else:
                # =====================================================
                # Local pyannote.audio
                # =====================================================
                print("Model: pyannote/speaker-diarization-3.1 (local)")
                print()
                
                # Initialize local diarizer
                print("Loading pyannote.audio diarization pipeline...")
                load_start = time.time()
                
                diarizer = SpeakerDiarizer(hf_token=HUGGINGFACE_TOKEN)
                
                load_time = time.time() - load_start
                print(f"  - Pipeline loaded in {load_time:.1f} seconds")
                print()
                
                # Run diarization
                print("Running speaker diarization...")
                print("-" * 40)
                
                segments = diarizer.diarize(vocals_path)
                
                print("-" * 40)
            
            api_time = time.time() - api_start
            print(f"Diarization completed in {api_time:.1f} seconds")
            print()
        
            if not segments:
                print("WARNING: No speakers detected in audio")
                print("  The audio may not contain speech or the model couldn't detect it.")
                diarization_segments = []
                video_info['diarization_segments'] = []
                video_info['num_speakers'] = 0
            else:
                # Convert to notebook's expected format
                speaker_set = set()
            
                for seg in segments:
                    speaker_set.add(seg["speaker"])
                    diarization_segments.append({
                        "speaker": seg["speaker"],
                        "start": seg["start"],
                        "end": seg["end"]
                    })
                
                # Save diarization to file
                diarization_file = job_dir / "diarization.json"
                with open(diarization_file, "w") as f:
                    json_module.dump(diarization_segments, f, indent=2)
                print(f"Saved diarization to: {diarization_file}")
                print()
            
                # Store in video_info for later cells
                video_info['diarization_segments'] = diarization_segments
                video_info['num_speakers'] = len(speaker_set)
            
                # Display summary
                print("Diarization Summary")
                print("-" * 40)
                print(f"Number of speakers detected: {len(speaker_set)}")
                print(f"Total segments: {len(diarization_segments)}")
                print()
            
                # Calculate per-speaker statistics
                speaker_stats = {}
                for seg in diarization_segments:
                    spk = seg['speaker']
                    duration = seg['end'] - seg['start']
                    if spk not in speaker_stats:
                        speaker_stats[spk] = {'count': 0, 'total_duration': 0.0}
                    speaker_stats[spk]['count'] += 1
                    speaker_stats[spk]['total_duration'] += duration
            
                print("Per-Speaker Statistics")
                print("-" * 40)
                for spk, stats in sorted(speaker_stats.items()):
                    print(f"  {spk}: {stats['count']} segments, {stats['total_duration']:.1f}s total")
                print()
            
                # Display first 20 segments
                print("Diarization Segments (first 20)")
                print("-" * 60)
                print(f"{'#':<4} {'Speaker':<12} {'Start':>10} {'End':>10} {'Duration':>10}")
                print("-" * 60)
            
                for i, seg in enumerate(diarization_segments[:20]):
                    duration = seg['end'] - seg['start']
                    start_str = f"{seg['start']:.2f}s"
                    end_str = f"{seg['end']:.2f}s"
                    dur_str = f"{duration:.2f}s"
                    print(f"{i:<4} {seg['speaker']:<12} {start_str:>10} {end_str:>10} {dur_str:>10}")
                
                if len(diarization_segments) > 20:
                    print(f"... ({len(diarization_segments) - 20} more segments)")
            
                print("-" * 60)
        
            print()
            print("=" * 60)
            print("Speaker diarization complete! Proceed to voice sample extraction.")
            print("=" * 60)
    
        except Exception as e:
            print(f"ERROR: Unexpected error during speaker diarization")
            print(f"  Exception type: {type(e).__name__}")
            print(f"  Details: {str(e)}")
            print()
            print("Fallback: Continuing without diarization (single-speaker mode)")
            diarization_segments = []
            video_info['diarization_segments'] = []
            video_info['num_speakers'] = 0

## Cell 8: Voice Sample Extraction

This cell extracts voice samples for each speaker identified during diarization. These samples are used by the local Chatterbox TTS model to clone each speaker's voice.

### Why Voice Samples?

Chatterbox (and similar voice cloning models) requires a reference audio sample of each speaker's voice to:

- **Learn voice characteristics**: Pitch, tone, speaking style
- **Enable voice cloning**: Generate synthesized speech that sounds like the original speaker

### What This Cell Does

1. **Groups diarization segments** by speaker ID
2. **Finds longest segment** for each speaker (best quality sample)
3. **Extracts samples** using ffmpeg at 24kHz mono (optimal for Chatterbox)
4. **Stores local paths** in `video_info['speaker_samples']`

### Sample Parameters

| Parameter | Value | Notes |
|-----------|-------|-------|
| Min Duration | 3 seconds | Segments shorter than this are skipped |
| Max Duration | 15 seconds | Longer segments are trimmed |
| Sample Rate | 24kHz | Chatterbox native rate |
| Channels | Mono | Required by Chatterbox |

### Expected Output

- Voice sample files for each speaker (e.g., `SPEAKER_00_sample.wav`)
- Summary table with duration and file size
- Local file paths stored for TTS synthesis

In [ ]:
# =============================================================================
# Voice Sample Extraction - Extract samples for each speaker for voice cloning
# =============================================================================

from collections import defaultdict


def extract_speaker_samples(
    audio_path: Path,
    output_dir: Path,
    segments: list[dict],
    target_duration: float = MAX_VOICE_SAMPLE_DURATION,
    min_duration: float = MIN_VOICE_SAMPLE_DURATION
) -> dict[str, Path]:
    """
    Extract voice samples for each speaker using ffmpeg.

    Groups segments by speaker ID, finds the longest segment for each speaker,
    and extracts a voice sample using ffmpeg.

    Args:
        audio_path: Path to the source audio file (vocals)
        output_dir: Directory to save extracted samples
        segments: List of diarization segments with 'speaker', 'start', 'end' keys
        target_duration: Maximum duration for each sample
        min_duration: Minimum segment duration required to extract

    Returns:
        Dict mapping speaker ID to the Path of the extracted sample file.
        Speakers with segments shorter than min_duration are skipped.
    """
    # Group segments by speaker
    speaker_segments: dict[str, list[dict]] = defaultdict(list)
    for seg in segments:
        speaker_id = seg.get("speaker", "SPEAKER_00")
        if speaker_id:  # Skip empty speaker IDs
            speaker_segments[speaker_id].append(seg)

    result: dict[str, Path] = {}

    for speaker_id, segs in speaker_segments.items():
        # Find the longest segment for this speaker
        longest_seg = max(segs, key=lambda s: s["end"] - s["start"])
        seg_duration = longest_seg["end"] - longest_seg["start"]

        # Skip if segment is too short
        if seg_duration < min_duration:
            print(f"  - {speaker_id}: Skipped (longest segment {seg_duration:.1f}s < min {min_duration}s)")
            continue

        # Calculate extraction duration (capped at target_duration)
        extract_duration = min(seg_duration, target_duration)

        # Output file path
        sample_path = output_dir / f"{speaker_id}_sample.wav"

        # Extract using ffmpeg at 24kHz mono (optimal for Chatterbox)
        cmd = [
            "ffmpeg",
            "-y",  # Overwrite output
            "-i", str(audio_path),
            "-ss", str(longest_seg["start"]),
            "-t", str(extract_duration),
            "-ar", "24000",  # 24kHz for Chatterbox
            "-ac", "1",      # Mono
            str(sample_path)
        ]

        try:
            subprocess.run(cmd, check=True, capture_output=True)
            result[speaker_id] = sample_path
            print(f"  - {speaker_id}: Extracted {extract_duration:.1f}s sample from segment at {longest_seg['start']:.1f}s")
        except subprocess.CalledProcessError as e:
            print(f"  - {speaker_id}: ffmpeg failed ({e.returncode})")
            continue

    return result


# =============================================================================
# Extract Voice Samples (Local - no Replicate upload needed)
# =============================================================================

speaker_samples = {}  # Will hold local paths to voice samples

print("=" * 60)
print("VOICE SAMPLE EXTRACTION (LOCAL)")
print("=" * 60)
print()

# Check prerequisites from previous cells
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('vocals_path') is None:
    print("ERROR: vocals_path not available. Run the Audio Separation cell first.")
elif video_info.get('diarization_segments') is None:
    print("ERROR: diarization_segments not available. Run the Speaker Diarization cell first.")
else:
    # === Skip check ===
    _skip = False
    _job_dir = video_info["vocals_path"].parent
    _samples_dir = _job_dir / "voice_samples"
    if _samples_dir.exists() and list(_samples_dir.glob("*.wav")):
        print(f"SKIPPING: Voice samples already exist")
        speaker_samples = {p.stem.replace("_sample", ""): p for p in _samples_dir.glob("*.wav")}
        video_info["speaker_samples"] = speaker_samples
        _skip = True

    if not _skip:
        try:
            vocals_path = video_info['vocals_path']
            job_dir = vocals_path.parent
            diarization_segments = video_info['diarization_segments']
            num_speakers = video_info.get('num_speakers', 0)
        
            print(f"Source audio: {vocals_path}")
            print(f"Number of speakers: {num_speakers}")
            print(f"Sample duration: {MIN_VOICE_SAMPLE_DURATION}-{MAX_VOICE_SAMPLE_DURATION} seconds")
            print(f"Output format: 24kHz mono WAV (optimal for Chatterbox)")
            print()
        
            if not diarization_segments:
                print("WARNING: No diarization segments available.")
                print("  Voice sample extraction requires speaker diarization.")
                print("  Pipeline will use single-speaker fallback mode.")
            else:
                # Extract voice samples using ffmpeg
                print("Extracting voice samples...")
                print("-" * 40)
            
                extract_start = time.time()
                speaker_samples = extract_speaker_samples(
                    audio_path=vocals_path,
                    output_dir=job_dir,
                    segments=diarization_segments,
                    target_duration=MAX_VOICE_SAMPLE_DURATION,
                    min_duration=MIN_VOICE_SAMPLE_DURATION
                )
                extract_time = time.time() - extract_start
            
                print("-" * 40)
                print(f"Extraction completed in {extract_time:.1f} seconds")
                print()
            
                if not speaker_samples:
                    print("WARNING: No voice samples extracted.")
                    print("  All segments may be too short for extraction.")
                    print("  Pipeline will use single-speaker fallback mode.")
                else:
                    # Store in video_info for later cells (local paths only)
                    video_info['speaker_samples'] = speaker_samples
                
                    # Display summary
                    print("Voice Sample Summary")
                    print("-" * 60)
                    print(f"{'Speaker':<15} {'Duration':>10} {'File Size':>12} {'Path':<30}")
                    print("-" * 60)
                
                    for speaker_id, sample_path in speaker_samples.items():
                        duration = get_audio_duration(sample_path)
                        size = sample_path.stat().st_size
                        print(f"{speaker_id:<15} {duration:>9.1f}s {format_file_size(size):>12} {sample_path.name:<30}")
                
                    print("-" * 60)
                    print()
                
                    # Timing summary
                    print("Timing")
                    print("-" * 40)
                    print(f"Sample extraction: {extract_time:.1f} seconds")
        
            print()
            print("=" * 60)
            print("Voice sample extraction complete! Proceed to transcription.")
            print("=" * 60)
    
        except subprocess.CalledProcessError as e:
            print(f"ERROR: ffmpeg extraction failed")
            print(f"  Return code: {e.returncode}")
            if e.stderr:
                print(f"  stderr: {e.stderr.decode()[:200]}")
    
        except Exception as e:
            print(f"ERROR: Unexpected error during voice sample extraction")
            print(f"  Exception type: {type(e).__name__}")
            print(f"  Details: {str(e)}")
            print()
            print("Fallback: Pipeline will use single-speaker mode.")
            speaker_samples = {}
            video_info['speaker_samples'] = {}


## Cell 9: Transcription

This cell transcribes the separated vocals using OpenAI's **Whisper API**. Whisper is a state-of-the-art speech recognition model that converts audio to text with timestamps.

### Whisper API Parameters

The API call uses these parameters:

| Parameter | Value | Description |
|-----------|-------|-------------|
| `model` | `whisper-1` | OpenAI's Whisper model (latest version) |
| `file` | audio file | The vocals.wav file from audio separation |
| `response_format` | `verbose_json` | Returns detailed segment information |
| `timestamp_granularities` | `["segment"]` | Include segment-level timestamps |

### Response Format

The `verbose_json` response includes:

```json
{
  "text": "Full transcription text...",
  "language": "en",
  "duration": 60.0,
  "segments": [
    {
      "id": 0,
      "start": 0.0,
      "end": 5.2,
      "text": "First segment text",
      "tokens": [...],
      "temperature": 0.0,
      ...
    },
    ...
  ]
}
```

### Segment Structure

Each segment contains:

| Field | Type | Description |
|-------|------|-------------|
| `start` | float | Start time in seconds |
| `end` | float | End time in seconds |
| `text` | string | Transcribed text for this segment |
| `tokens` | array | Token IDs (for advanced use) |
| `temperature` | float | Sampling temperature used |

### Output

- **transcription_segments**: List of segments with timestamps and text
- **Detected language**: Whisper auto-detects the source language
- **Segment count**: Total number of segments transcribed
- **Segment table**: Formatted display showing index, timing, and text

### Timing

Processing time depends on audio length and server load:
- 60 seconds audio: ~5-15 seconds
- 5 minutes audio: ~30-60 seconds

In [ ]:
# =============================================================================
# Transcription - Convert speech to text
# =============================================================================

# =============================================================================
# Transcription
# =============================================================================

transcription_segments = []  # Will hold the parsed transcription segments

print("=" * 60)
print(f"TRANSCRIPTION ({'RunPod Faster Whisper' if USE_RUNPOD_ENDPOINTS else 'OpenAI Whisper API'})")
print("=" * 60)
print()

# Check prerequisites from previous cells
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('vocals_path') is None:
    print("ERROR: vocals_path not available. Run the Audio Separation cell first.")
elif not USE_RUNPOD_ENDPOINTS and openai_client is None:
    print("ERROR: OpenAI client not initialized. Check OPENAI_API_KEY in setup cell.")
else:
    # === Skip check ===
    _skip = False
    _job_dir = video_info["vocals_path"].parent
    if (_job_dir / "transcription.json").exists():
        print(f"SKIPPING: Transcription already exists")
        import json as _jm
        with open(_job_dir / "transcription.json") as _f:
            transcription_segments = _jm.load(_f)
        video_info["transcription_segments"] = transcription_segments
        # Also load detected language if available
        if (_job_dir / "language.txt").exists():
            video_info["detected_language"] = (_job_dir / "language.txt").read_text().strip()
        _skip = True

    if not _skip:
        try:
            vocals_path = video_info['vocals_path']
            job_dir = vocals_path.parent
        
            print(f"Source audio: {vocals_path}")
            print(f"File size: {format_file_size(vocals_path.stat().st_size)}")
            print()
        
            api_start = time.time()
            
            if USE_RUNPOD_ENDPOINTS:
                # =====================================================
                # RunPod Faster Whisper Endpoint
                # =====================================================
                print("Using RunPod Faster Whisper endpoint...")
                print()
                
                # Read audio and encode to base64
                with open(vocals_path, "rb") as f:
                    audio_base64 = base64.b64encode(f.read()).decode("utf-8")
                print(f"  Audio encoded: {len(audio_base64) / 1e6:.1f} MB base64")
                
                # Call RunPod endpoint
                result = run_runpod_sync("whisper", {
                    "audio_base64": audio_base64,
                    "word_timestamps": True,
                })
                
                if result.get("status") == "error":
                    raise RuntimeError(f"Whisper failed: {result.get('error', 'Unknown error')}")
                
                # Extract results
                raw_segments = result.get("segments", [])
                detected_language = result.get("language", "unknown")
                full_text = result.get("text", "")
                audio_duration = result.get("duration", 0.0)
                
                # Parse segments
                for seg in raw_segments:
                    transcription_segments.append({
                        'start': seg.get('start', 0.0),
                        'end': seg.get('end', 0.0),
                        'text': seg.get('text', '').strip(),
                        'duration': seg.get('end', 0.0) - seg.get('start', 0.0)
                    })
                
            else:
                # =====================================================
                # OpenAI Whisper API
                # =====================================================
                print("Model: whisper-1")
                print("Response format: verbose_json")
                print()
                
                # Call OpenAI Whisper API
                print("Calling OpenAI Whisper API...")
                print("-" * 40)
                
                with open(vocals_path, "rb") as audio_file:
                    transcript = openai_client.audio.transcriptions.create(
                        model="whisper-1",
                        file=audio_file,
                        response_format="verbose_json",
                        timestamp_granularities=["segment"]
                    )
                
                print("-" * 40)
                
                # Extract metadata from response
                detected_language = getattr(transcript, 'language', 'unknown')
                audio_duration = getattr(transcript, 'duration', 0.0)
                full_text = getattr(transcript, 'text', '')
                
                # Parse segments from response
                raw_segments = getattr(transcript, 'segments', []) or []
                
                for seg in raw_segments:
                    # Handle both dict and object access patterns
                    if isinstance(seg, dict):
                        start = seg.get('start', 0.0)
                        end = seg.get('end', 0.0)
                        text = seg.get('text', '').strip()
                    else:
                        start = getattr(seg, 'start', 0.0)
                        end = getattr(seg, 'end', 0.0)
                        text = getattr(seg, 'text', '').strip()
                    
                    transcription_segments.append({
                        'start': start,
                        'end': end,
                        'text': text,
                        'duration': end - start
                    })
            
            api_time = time.time() - api_start
            print(f"Transcription completed in {api_time:.1f} seconds")
            print()
            
            # Save transcription to file
            transcription_file = job_dir / "transcription.json"
            with open(transcription_file, "w") as f:
                json_module.dump(transcription_segments, f, indent=2)
            print(f"Saved transcription to: {transcription_file}")
            
            # Save detected language
            language_file = job_dir / "language.txt"
            language_file.write_text(detected_language)
        
            # Store in video_info for later cells
            video_info['transcription_segments'] = transcription_segments
            video_info['detected_language'] = detected_language
            video_info['full_transcript'] = full_text if 'full_text' in dir() else ' '.join(s['text'] for s in transcription_segments)
        
            # Display summary
            print()
            print("Transcription Summary")
            print("-" * 40)
            print(f"Detected language: {detected_language}")
            if 'audio_duration' in dir() and audio_duration:
                print(f"Audio duration: {audio_duration:.2f} seconds")
            print(f"Total segments: {len(transcription_segments)}")
            print()
        
            # Display full transcript preview
            full_text_display = video_info.get('full_transcript', '')
            print("Full Transcript (preview)")
            print("-" * 40)
            preview = full_text_display[:500] + "..." if len(full_text_display) > 500 else full_text_display
            print(preview)
            print()
        
            # Display segment table (first 20)
            print("Transcription Segments (first 20)")
            print("-" * 80)
            print(f"{'#':<4} {'Start':>8} {'End':>8} {'Text':<56}")
            print("-" * 80)
        
            for i, seg in enumerate(transcription_segments[:20]):
                start_str = f"{seg['start']:.2f}s"
                end_str = f"{seg['end']:.2f}s"
                # Truncate text for display
                text_display = seg['text'][:53] + "..." if len(seg['text']) > 53 else seg['text']
                print(f"{i:<4} {start_str:>8} {end_str:>8} {text_display:<56}")
            
            if len(transcription_segments) > 20:
                print(f"... ({len(transcription_segments) - 20} more segments)")
        
            print("-" * 80)
            print()
        
            # Timing summary
            print("Timing")
            print("-" * 40)
            print(f"Transcription: {api_time:.1f} seconds")
        
            print()
            print("=" * 60)
            print("Transcription complete! Proceed to translation.")
            print("=" * 60)
    
        except Exception as e:
            print(f"ERROR: Transcription failed")
            print(f"  Exception type: {type(e).__name__}")
            print(f"  Details: {str(e)}")
            import traceback
            traceback.print_exc()

## Cell 10: Translation

This cell translates the transcribed segments using Replicate's **Llama 3.1 405B** model. The LLM approach provides context-aware translation that maintains consistency across segments.

### Why Use an LLM for Translation?

Traditional machine translation (like Google Translate) works on individual sentences without context. LLM-based translation offers:

- **Context awareness**: The model sees surrounding segments, maintaining narrative flow
- **Consistent terminology**: Technical terms and names are translated consistently
- **Natural phrasing**: Translations sound more natural and conversational
- **Tone preservation**: The speaker's style and emotion are maintained

### Batching Strategy

For long videos, translation is done in batches to stay within the LLM's context limits:

| Parameter | Value | Description |
|-----------|-------|-------------|
| `LLM_MAX_SEGMENTS_PER_BATCH` | 50 | Maximum segments per LLM call |
| `LLM_BATCH_OVERLAP` | 5 | Overlapping segments for context continuity |
| `LLM_MAX_TOKENS` | 4096 | Maximum response tokens |
| `LLM_TEMPERATURE` | 0.3 | Lower temperature for consistent translations |

### Prompt Format

Segments are sent as numbered lines:
```
0: First segment text
1: Second segment text
...
```

The LLM returns translations in the same numbered format, which are then parsed and mapped back to segments.

### Output

- **translated_segments**: List of segments with both original and translated text
- **Comparison table**: Side-by-side display of original vs. translated text
- **Batch count**: Number of LLM calls made (for long videos)
- **API timing**: Processing time for translation

In [ ]:
# =============================================================================
# Translation - Translate segments using LLM
# =============================================================================


def translate_batch_llm_replicate(transcript_lines: list[str], target_lang_name: str) -> dict[int, str]:
    """
    Translate a batch of transcript lines using Replicate LLM.

    Args:
        transcript_lines: List of "INDEX: text" formatted lines
        target_lang_name: Human-readable language name (e.g., "Spanish")

    Returns:
        Dict mapping segment index to translated text
    """
    transcript = "\n".join(transcript_lines)

    prompt = f"""Translate the following numbered transcript lines to {target_lang_name}.

IMPORTANT RULES:
1. Maintain the exact same line numbers in your output
2. Only output the translated text, preserving the "NUMBER: text" format
3. Keep translations natural and conversational
4. Maintain consistent terminology throughout
5. Preserve the speaker's tone and style
6. Do not add any explanations or notes

Transcript:
{transcript}

Translate each line to {target_lang_name}, keeping the same numbered format:"""

    output = replicate.run(
        REPLICATE_MODEL_LLAMA,
        input={
            "prompt": prompt,
            "max_tokens": LLM_MAX_TOKENS,
            "temperature": LLM_TEMPERATURE,
        }
    )

    # Collect streaming output
    if hasattr(output, '__iter__') and not isinstance(output, str):
        response_text = "".join(str(chunk) for chunk in output)
    else:
        response_text = str(output)

    # Parse the numbered response back into translations
    translations = {}
    for line in response_text.strip().split("\n"):
        line = line.strip()
        if not line or ":" not in line:
            continue

        parts = line.split(":", 1)
        try:
            idx_str = parts[0].strip()
            idx = int(idx_str)
            translations[idx] = parts[1].strip()
        except ValueError:
            continue

    return translations


def translate_batch_llm_vllm(transcript_lines: list[str], target_lang_name: str) -> dict[int, str]:
    """
    Translate a batch of transcript lines using RunPod vLLM endpoint.

    Args:
        transcript_lines: List of "INDEX: text" formatted lines
        target_lang_name: Human-readable language name (e.g., "Spanish")

    Returns:
        Dict mapping segment index to translated text
    """
    transcript = "\n".join(transcript_lines)

    prompt = f"""Translate the following numbered transcript lines to {target_lang_name}.

IMPORTANT RULES:
1. Maintain the exact same line numbers in your output
2. Only output the translated text, preserving the "NUMBER: text" format
3. Keep translations natural and conversational
4. Maintain consistent terminology throughout
5. Preserve the speaker's tone and style
6. Do not add any explanations or notes

Transcript:
{transcript}

Translate each line to {target_lang_name}, keeping the same numbered format:"""

    # Use the pre-initialized vllm_client (OpenAI-compatible)
    response = vllm_client.chat.completions.create(
        model="Qwen/Qwen3-32B",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=LLM_MAX_TOKENS,
        temperature=LLM_TEMPERATURE,
    )
    
    response_text = response.choices[0].message.content

    # Parse the numbered response back into translations
    translations = {}
    for line in response_text.strip().split("\n"):
        line = line.strip()
        if not line or ":" not in line:
            continue

        parts = line.split(":", 1)
        try:
            idx_str = parts[0].strip()
            idx = int(idx_str)
            translations[idx] = parts[1].strip()
        except ValueError:
            continue

    return translations


# Select the translation function based on mode
translate_batch_llm = translate_batch_llm_vllm if USE_RUNPOD_ENDPOINTS else translate_batch_llm_replicate


# =============================================================================
# Translation
# =============================================================================

translated_segments = []  # Will hold the translated segments

print("=" * 60)
print(f"TRANSLATION ({'RunPod vLLM (Qwen3-32B)' if USE_RUNPOD_ENDPOINTS else 'Replicate LLM (Llama)'})")
print("=" * 60)
print()

# Check prerequisites from previous cells
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('transcription_segments') is None:
    print("ERROR: transcription_segments not available. Run the Transcription cell first.")
else:
    # === Skip check ===
    _skip = False
    _job_dir = video_info["video_path"].parent
    if (_job_dir / "translation.json").exists():
        print(f"SKIPPING: Translation already exists")
        import json as _jm
        with open(_job_dir / "translation.json") as _f:
            translated_segments = _jm.load(_f)
        video_info["translated_segments"] = translated_segments
        _skip = True

    if not _skip:
        try:
            transcription_segments = video_info['transcription_segments']
            target_lang_name = LANG_NAMES.get(TARGET_LANGUAGE, TARGET_LANGUAGE)
        
            print(f"Source segments: {len(transcription_segments)}")
            print(f"Target language: {TARGET_LANGUAGE} ({target_lang_name})")
            if USE_RUNPOD_ENDPOINTS:
                print(f"Model: Qwen/Qwen3-32B (via RunPod vLLM)")
            else:
                print(f"Model: {REPLICATE_MODEL_LLAMA}")
            print(f"Batch size: {LLM_MAX_SEGMENTS_PER_BATCH} segments (overlap: {LLM_BATCH_OVERLAP})")
            print()
        
            if not transcription_segments:
                print("WARNING: No transcription segments to translate.")
                translated_segments = []
            else:
                # Build numbered transcript for the LLM
                transcript_lines = []
                segment_indices = []  # Track which segments have text
                for i, seg in enumerate(transcription_segments):
                    text = seg.get("text", "").strip()
                    if text:
                        transcript_lines.append(f"{i}: {text}")
                        segment_indices.append(i)
            
                if not transcript_lines:
                    print("WARNING: No text content in segments to translate.")
                    translated_segments = [{
                        "start": s.get("start", 0.0),
                        "end": s.get("end", 0.0),
                        "duration": s.get("duration", 0.0),
                        "original_text": s.get("text", ""),
                        "translated_text": ""
                    } for s in transcription_segments]
                else:
                    print(f"Segments with text: {len(transcript_lines)}")
                    print()
                
                    # Determine batching strategy
                    all_translations = {}
                    batch_count = 0
                
                    if len(transcript_lines) <= LLM_MAX_SEGMENTS_PER_BATCH:
                        # Single batch - translate all at once
                        print("Translating all segments in a single batch...")
                        print("-" * 40)
                    
                        api_start = time.time()
                        batch_translations = translate_batch_llm(transcript_lines, target_lang_name)
                        api_time = time.time() - api_start
                    
                        all_translations = batch_translations
                        batch_count = 1
                    
                        print("-" * 40)
                        print(f"Batch completed in {api_time:.1f} seconds")
                        print(f"Translated {len(batch_translations)}/{len(transcript_lines)} segments")
                    else:
                        # Multiple batches with overlap for context continuity
                        total_batches = (len(transcript_lines) + LLM_MAX_SEGMENTS_PER_BATCH - LLM_BATCH_OVERLAP - 1) // (LLM_MAX_SEGMENTS_PER_BATCH - LLM_BATCH_OVERLAP)
                        print(f"Translating {len(transcript_lines)} segments in {total_batches} batches...")
                        print("-" * 40)
                    
                        api_start = time.time()
                    
                        for batch_start in range(0, len(transcript_lines), LLM_MAX_SEGMENTS_PER_BATCH - LLM_BATCH_OVERLAP):
                            batch_end = min(batch_start + LLM_MAX_SEGMENTS_PER_BATCH, len(transcript_lines))
                            batch_lines = transcript_lines[batch_start:batch_end]
                        
                            batch_count += 1
                            print(f"  Batch {batch_count}/{total_batches}: segments {batch_start}-{batch_end-1}")
                        
                            batch_translations = translate_batch_llm(batch_lines, target_lang_name)
                        
                            # Merge translations (later batches overwrite overlap regions)
                            all_translations.update(batch_translations)
                            print(f"    -> Translated {len(batch_translations)} segments")
                    
                        api_time = time.time() - api_start
                        print("-" * 40)
                        print(f"All batches completed in {api_time:.1f} seconds")
                
                    print()
                
                    # Build translated segments list
                    for i, seg in enumerate(transcription_segments):
                        original_text = seg.get("text", "")
                        translated_text = all_translations.get(i, original_text)  # Fallback to original
                    
                        translated_segments.append({
                            "start": seg.get("start", 0.0),
                            "end": seg.get("end", 0.0),
                            "duration": seg.get("duration", 0.0),
                            "original_text": original_text,
                            "translated_text": translated_text
                        })
                    
                    # Save translation to file
                    translation_file = _job_dir / "translation.json"
                    with open(translation_file, "w") as f:
                        json_module.dump(translated_segments, f, indent=2)
                    print(f"Saved translation to: {translation_file}")
                
                    # Store in video_info for later cells
                    video_info['translated_segments'] = translated_segments
                
                    # Display summary
                    print()
                    print("Translation Summary")
                    print("-" * 40)
                    print(f"Total segments: {len(translated_segments)}")
                    print(f"Segments translated: {len(all_translations)}")
                    print(f"Batches used: {batch_count}")
                    print()
                
                    # Display translation comparison table (first 20)
                    print("Translation Comparison (first 20)")
                    print("-" * 100)
                    print(f"{'#':<4} {'Original Text':<45} {'Translated Text':<45}")
                    print("-" * 100)
                
                    for i, seg in enumerate(translated_segments[:20]):
                        orig = seg['original_text']
                        trans = seg['translated_text']
                        # Truncate for display
                        orig_display = orig[:42] + "..." if len(orig) > 42 else orig
                        trans_display = trans[:42] + "..." if len(trans) > 42 else trans
                        print(f"{i:<4} {orig_display:<45} {trans_display:<45}")
                    
                    if len(translated_segments) > 20:
                        print(f"... ({len(translated_segments) - 20} more segments)")
                
                    print("-" * 100)
                    print()
                
                    # Timing summary
                    print("Timing")
                    print("-" * 40)
                    print(f"API calls: {batch_count}")
                    print(f"Total time: {api_time:.1f} seconds")
                    if batch_count > 0:
                        print(f"Average per batch: {api_time/batch_count:.1f} seconds")
        
            print()
            print("=" * 60)
            print("Translation complete! Proceed to speech synthesis.")
            print("=" * 60)
    
        except Exception as e:
            print(f"ERROR: Translation failed")
            print(f"  Exception type: {type(e).__name__}")
            print(f"  Details: {str(e)}")
            import traceback
            traceback.print_exc()

## Cell 11: Single-Speaker Synthesis

This cell synthesizes all translated text using a **single voice**. Use this mode when `USE_MULTI_SPEAKER = False` or when speaker diarization was skipped/failed.

### When to Use Single-Speaker Mode

- **Single presenter**: Educational videos, tutorials, monologues
- **Narration**: Documentaries with voiceover
- **Fallback**: When multi-speaker diarization fails

### What This Cell Does

1. **Loads local Chatterbox model** once
2. **Processes each segment** individually
3. **Generates speech** at natural pace (no time-stretching)
4. **Combines segments** with overlap prevention
5. **Saves combined output** as single WAV file

### Model Details

| Property | Value |
|----------|-------|
| Model | ChatterboxMultilingualTTS (local) |
| Execution | Local GPU/CPU |
| VRAM Usage | ~4-6GB |
| Output Sample Rate | 24kHz |

### Timing Approach

Audio is generated at **natural speaking pace** for best quality:
- Each segment starts at max(original_timestamp, previous_segment_end)
- This prevents overlap while preserving natural speech
- **Output may be longer than original video** - this is expected
- Final video merge will extend video to match audio length

### Expected Output

- `translated_audio_single.wav` - Combined synthesized speech at natural pace
- Summary showing duration comparison with original video

In [ ]:
# =============================================================================
# Single-Speaker Synthesis - Synthesize each segment with one voice
# =============================================================================

import numpy as np
if not USE_RUNPOD_ENDPOINTS:
    import torch

# =============================================================================
# Single-Speaker Synthesis (Natural Timing)
# =============================================================================

single_speaker_output = None  # Will hold the path to synthesized audio

print("=" * 60)
print(f"SINGLE-SPEAKER SYNTHESIS ({'RunPod Chatterbox' if USE_RUNPOD_ENDPOINTS else 'Local Chatterbox'})")
print("=" * 60)
print()

# Check if this mode should run
if USE_MULTI_SPEAKER:
    print("Skipping: USE_MULTI_SPEAKER is True")
    print("  This cell only runs when USE_MULTI_SPEAKER is False.")
    print("  Proceed to the Multi-Speaker Synthesis cell instead.")
# Check prerequisites from previous cells
elif video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('translated_segments') is None:
    print("ERROR: translated_segments not available. Run the Translation cell first.")
else:
    # === Skip check ===
    _skip = False
    _job_dir = video_info["video_path"].parent
    if (_job_dir / "translated_audio_single.wav").exists():
        print(f"SKIPPING: Single-speaker audio already exists")
        single_speaker_output = _job_dir / "translated_audio_single.wav"
        video_info["synthesized_audio_path"] = single_speaker_output
        _skip = True

    if not _skip:
        try:
            translated_segments = video_info['translated_segments']
            job_dir = video_info['video_path'].parent
        
            print(f"Translated segments: {len(translated_segments)}")
            print()
        
            # Get voice sample path - use first available or fall back to vocals
            speaker_samples = video_info.get('speaker_samples', {})
            vocals_path = video_info.get('vocals_path')
        
            voice_sample_path = None
            voice_source = None
        
            if speaker_samples:
                # Use first available voice sample
                first_speaker = sorted(speaker_samples.keys())[0]
                voice_sample_path = speaker_samples[first_speaker]
                voice_source = f"voice sample from {first_speaker}"
                print(f"Using voice sample: {first_speaker}")
            elif vocals_path and vocals_path.exists():
                # Use vocals as fallback voice sample
                voice_sample_path = vocals_path
                voice_source = "original vocals (fallback)"
                print("No voice samples available, using vocals as reference...")
            else:
                raise ValueError("No voice sample or vocals available for synthesis")
        
            print(f"Voice source: {voice_source}")
            print(f"Voice sample path: {voice_sample_path}")
            print()
            
            # Process each segment
            total_segments = len(translated_segments)
            segment_audios: list[tuple[float, np.ndarray]] = []  # (original_start, audio)
            segment_info = []  # Track for display
            
            print("Processing segments...")
            print("-" * 60)
            
            total_gen_start = time.time()
            
            if USE_RUNPOD_ENDPOINTS:
                # =====================================================
                # RunPod Chatterbox Endpoint
                # =====================================================
                print("Using RunPod Chatterbox endpoint...")
                print()
                
                # Read voice sample and encode to base64
                with open(voice_sample_path, "rb") as f:
                    voice_sample_base64 = base64.b64encode(f.read()).decode("utf-8")
                print(f"  Voice sample encoded: {len(voice_sample_base64) / 1e6:.1f} MB base64")
                print()
                
                sample_rate = CHATTERBOX_SAMPLE_RATE  # Use config constant
                
                for i, seg in enumerate(translated_segments):
                    text = seg.get('translated_text', '').strip()
                    
                    if not text:
                        segment_info.append((i, seg['start'], seg['end'], '(empty)'))
                        continue
                    
                    print(f"Segment {i+1}/{total_segments}: ({seg['start']:.1f}s - {seg['end']:.1f}s)")
                    
                    try:
                        segment_start = time.time()
                        
                        # Call RunPod endpoint
                        result = run_runpod_sync("chatterbox", {
                            "text": text,
                            "voice_sample_base64": voice_sample_base64,
                            "language": TARGET_LANGUAGE,
                            "cfg_weight": CHATTERBOX_CFG_WEIGHT,
                            "exaggeration": CHATTERBOX_EXAGGERATION,
                        })
                        
                        if result.get("status") == "error":
                            raise RuntimeError(result.get("error", "Unknown error"))
                        
                        # Decode audio from base64
                        audio_bytes = base64.b64decode(result["audio_base64"])
                        
                        # Read as wav
                        import io
                        import soundfile as sf
                        audio_np, sr = sf.read(io.BytesIO(audio_bytes))
                        sample_rate = sr
                        
                        # Normalize
                        max_val = np.max(np.abs(audio_np))
                        if max_val > 0.99:
                            audio_np = audio_np * 0.99 / max_val
                        
                        segment_audios.append((seg['start'], audio_np))
                        segment_info.append((i, seg['start'], seg['end'], 
                                           text[:40] + '...' if len(text) > 40 else text))
                        
                        segment_time = time.time() - segment_start
                        audio_duration = len(audio_np) / sample_rate
                        print(f"  -> Generated {audio_duration:.1f}s audio in {segment_time:.1f}s")
                        
                    except Exception as e:
                        print(f"  -> FAILED: {str(e)[:50]}")
                        segment_info.append((i, seg['start'], seg['end'], '(FAILED)'))
                        continue
                
                load_time = 0.0  # No model loading for RunPod
                
            else:
                # =====================================================
                # Local Chatterbox TTS
                # =====================================================
                device = "cuda" if torch.cuda.is_available() else "cpu"
                print(f"Loading Chatterbox multilingual model on {device.upper()}...")
                load_start = time.time()
                
                model = ChatterboxMultilingualTTS.from_pretrained(device=device)
                sample_rate = model.sr
                
                load_time = time.time() - load_start
                print(f"  - Model loaded in {load_time:.1f} seconds")
                print()
                
                for i, seg in enumerate(translated_segments):
                    text = seg.get('translated_text', '').strip()
                    
                    if not text:
                        segment_info.append((i, seg['start'], seg['end'], '(empty)'))
                        continue
                    
                    print(f"Segment {i+1}/{total_segments}: ({seg['start']:.1f}s - {seg['end']:.1f}s)")
                    
                    try:
                        segment_start = time.time()
                        
                        wav = model.generate(
                            text,
                            audio_prompt_path=str(voice_sample_path),
                            language_id=TARGET_LANGUAGE,
                        )
                        
                        # Convert to numpy
                        audio_np = wav.squeeze().cpu().numpy()
                        
                        # Normalize
                        max_val = np.max(np.abs(audio_np))
                        if max_val > 0.99:
                            audio_np = audio_np * 0.99 / max_val
                        
                        segment_audios.append((seg['start'], audio_np))
                        segment_info.append((i, seg['start'], seg['end'], 
                                           text[:40] + '...' if len(text) > 40 else text))
                        
                        segment_time = time.time() - segment_start
                        audio_duration = len(audio_np) / sample_rate
                        print(f"  -> Generated {audio_duration:.1f}s audio in {segment_time:.1f}s")
                        
                    except Exception as e:
                        print(f"  -> FAILED: {str(e)[:50]}")
                        segment_info.append((i, seg['start'], seg['end'], '(FAILED)'))
                        continue
            
            print("-" * 60)
            total_gen_time = time.time() - total_gen_start
            print(f"All segments processed in {total_gen_time:.1f} seconds")
            print()
            
            if not segment_audios:
                print("ERROR: No segments were successfully synthesized!")
            else:
                # Combine all segments with adjusted timing to prevent overlap
                print("Combining segments with natural timing...")
                
                positioned_audios: list[tuple[float, np.ndarray]] = []
                current_end_time = 0.0
                
                for original_start, audio in segment_audios:
                    audio_duration = len(audio) / sample_rate
                    
                    # Start at the later of: original timestamp or when previous segment ends
                    actual_start = max(original_start, current_end_time)
                    positioned_audios.append((actual_start, audio))
                    
                    # Update end time for next segment
                    current_end_time = actual_start + audio_duration
                
                # Output length based on actual audio content
                actual_total_duration = current_end_time
                output_length = int(actual_total_duration * sample_rate) + sample_rate  # +1s buffer
                output_audio = np.zeros(output_length)
                
                # Place audio at calculated positions
                for start_time, audio in positioned_audios:
                    start_sample = int(start_time * sample_rate)
                    end_sample = start_sample + len(audio)
                    
                    if end_sample > output_length:
                        audio = audio[:output_length - start_sample]
                    
                    if start_sample < output_length:
                        output_audio[start_sample:start_sample + len(audio)] += audio
                
                # Normalize to prevent clipping
                max_val = np.max(np.abs(output_audio))
                if max_val > 0.99:
                    output_audio = output_audio * 0.99 / max_val
                
                # Trim trailing silence
                threshold = 0.001
                non_zero_indices = np.where(np.abs(output_audio) > threshold)[0]
                if len(non_zero_indices) > 0:
                    last_sound = non_zero_indices[-1]
                    trim_point = min(last_sound + int(0.5 * sample_rate), len(output_audio))
                    output_audio = output_audio[:trim_point]
                
                # Save the combined audio
                single_speaker_output = job_dir / "translated_audio_single.wav"
                write_wav_file(single_speaker_output, sample_rate, output_audio)
                
                # Store in video_info for later cells
                video_info['single_speaker_output'] = single_speaker_output
                video_info['synthesized_audio_path'] = single_speaker_output
                
                # Get output file info
                output_size = single_speaker_output.stat().st_size
                output_duration = get_audio_duration(single_speaker_output)
                
                # Get original video duration for comparison
                original_duration = video_info.get('actual_duration', video_info.get('duration', 0))
                
                print()
                print("Single-Speaker Synthesis Summary")
                print("-" * 40)
                print(f"Output file: {single_speaker_output}")
                print(f"File size: {format_file_size(output_size)}")
                print(f"Duration: {format_duration(int(output_duration))} ({output_duration:.2f} seconds)")
                print(f"Original video: {format_duration(int(original_duration))} ({original_duration:.2f} seconds)")
                if output_duration > original_duration:
                    print(f"Note: Audio is {output_duration - original_duration:.1f}s longer than original video")
                print(f"Sample rate: {sample_rate} Hz")
                print(f"Segments synthesized: {len(segment_audios)}/{total_segments}")
                print()
                
                # Timing summary
                print("Timing")
                print("-" * 40)
                if load_time > 0:
                    print(f"Model load: {load_time:.1f} seconds")
                print(f"Total generation: {total_gen_time:.1f} seconds")
                if len(segment_audios) > 0:
                    print(f"Average per segment: {total_gen_time/len(segment_audios):.1f} seconds")
                if output_duration > 0:
                    rtf = total_gen_time / output_duration
                    print(f"Real-time factor: {rtf:.2f}x")
    
        except Exception as e:
            print(f"ERROR: Single-speaker synthesis failed")
            print(f"  Exception type: {type(e).__name__}")
            print(f"  Details: {str(e)}")
            import traceback
            traceback.print_exc()

print()
print("=" * 60)
if single_speaker_output and single_speaker_output.exists():
    print("Single-speaker synthesis complete! Proceed to audio mixing.")
elif USE_MULTI_SPEAKER:
    print("Single-speaker mode skipped. Proceed to multi-speaker synthesis.")
else:
    print("Single-speaker synthesis failed. Check errors above.")
print("=" * 60)

## Cell 12: Multi-Speaker Synthesis

This cell synthesizes each translated segment using the **matched speaker's voice**. Use this mode when `USE_MULTI_SPEAKER = True` and speaker diarization has successfully identified multiple speakers.

### When to Use Multi-Speaker Mode

- **Interviews**: Different voices for interviewer and interviewee
- **Podcasts**: Multiple hosts or guests
- **Panel discussions**: Each participant gets their voice cloned

### What This Cell Does

1. **Loads local Chatterbox model** once (shared across all segments)
2. **Matches each segment** to a speaker via diarization timestamps
3. **Synthesizes each segment** with the matched speaker's voice sample
4. **Generates speech** at natural pace (no time-stretching)
5. **Combines segments** with overlap prevention
6. **Saves combined output** as single WAV file

### Model Details

| Property | Value |
|----------|-------|
| Model | ChatterboxMultilingualTTS (local) |
| Execution | Local GPU/CPU |
| VRAM Usage | ~4-6GB |
| Output Sample Rate | 24kHz |

### Timing Approach

Audio is generated at **natural speaking pace** for best quality:
- Each segment starts at max(original_timestamp, previous_segment_end)
- This prevents overlap while preserving natural speech
- **Output may be longer than original video** - this is expected
- Final video merge will extend video to match audio length

### Speaker Matching Algorithm

For each transcription segment, the algorithm:
1. Compares segment time range with diarization segments
2. Calculates overlap duration with each speaker's segments
3. Assigns the speaker with maximum overlap
4. Falls back to first speaker if no overlap found

### Expected Output

- Segment-to-speaker mapping table
- `translated_audio_multi.wav` - Combined synthesized speech at natural pace
- Per-segment timing and overall statistics

In [ ]:
# =============================================================================
# Multi-Speaker Synthesis - Synthesize each segment with matched speaker voice
# =============================================================================

import numpy as np
if not USE_RUNPOD_ENDPOINTS:
    import torch

# =============================================================================
# Multi-Speaker Synthesis (Natural Timing)
# =============================================================================

multi_speaker_output = None  # Will hold the path to synthesized audio

print("=" * 60)
print(f"MULTI-SPEAKER SYNTHESIS ({'RunPod Chatterbox' if USE_RUNPOD_ENDPOINTS else 'Local Chatterbox'})")
print("=" * 60)
print()

# Check if this mode should run
if not USE_MULTI_SPEAKER:
    print("Skipping: USE_MULTI_SPEAKER is False")
    print("  This cell only runs when USE_MULTI_SPEAKER is True.")
    print("  The single-speaker synthesis cell should have run instead.")
# Check prerequisites from previous cells
elif video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('translated_segments') is None:
    print("ERROR: translated_segments not available. Run the Translation cell first.")
elif video_info.get('diarization_segments') is None:
    print("ERROR: diarization_segments not available. Run the Speaker Diarization cell first.")
elif video_info.get('speaker_samples') is None or len(video_info.get('speaker_samples', {})) == 0:
    print("ERROR: speaker_samples not available or empty. Run the Voice Sample Extraction cell first.")
    print("  Falling back to single-speaker mode may be necessary.")
else:
    # === Skip check ===
    _skip = False
    _job_dir = video_info["video_path"].parent
    if (_job_dir / "translated_audio_multi.wav").exists():
        print(f"SKIPPING: Multi-speaker audio already exists")
        multi_speaker_output = _job_dir / "translated_audio_multi.wav"
        video_info["synthesized_audio_path"] = multi_speaker_output
        _skip = True

    if not _skip:
        try:
            translated_segments = video_info['translated_segments']
            diarization_segments = video_info['diarization_segments']
            speaker_samples = video_info['speaker_samples']  # Local paths
            job_dir = video_info['video_path'].parent
        
            print(f"Translated segments: {len(translated_segments)}")
            print(f"Diarization segments: {len(diarization_segments)}")
            print(f"Speaker samples: {len(speaker_samples)}")
            print(f"  Speakers: {', '.join(sorted(speaker_samples.keys()))}")
            print()
        
            # Get fallback voice sample (first available speaker)
            fallback_speaker = sorted(speaker_samples.keys())[0]
            fallback_sample = speaker_samples[fallback_speaker]
        
            def match_segment_to_speaker(seg_start: float, seg_end: float) -> str:
                """Find the speaker with maximum overlap for a given segment time range."""
                best_speaker = fallback_speaker
                best_overlap = 0.0
            
                for diar_seg in diarization_segments:
                    # Calculate overlap between transcription segment and diarization segment
                    overlap_start = max(seg_start, diar_seg['start'])
                    overlap_end = min(seg_end, diar_seg['end'])
                    overlap = max(0.0, overlap_end - overlap_start)
                
                    if overlap > best_overlap:
                        best_overlap = overlap
                        best_speaker = diar_seg['speaker']
            
                return best_speaker
            
            # Process each segment and collect audio with original start times
            total_segments = len(translated_segments)
            segment_audios: list[tuple[float, np.ndarray]] = []  # (original_start, audio)
            segment_info = []  # Track speaker assignments for display
            
            print("Processing segments...")
            print("-" * 60)
            
            total_gen_start = time.time()
            
            if USE_RUNPOD_ENDPOINTS:
                # =====================================================
                # RunPod Chatterbox Endpoint (Multi-Speaker)
                # =====================================================
                print("Using RunPod Chatterbox endpoint...")
                print()
                
                # Pre-encode all voice samples
                voice_samples_base64 = {}
                for spk, sample_path in speaker_samples.items():
                    with open(sample_path, "rb") as f:
                        voice_samples_base64[spk] = base64.b64encode(f.read()).decode("utf-8")
                print(f"  Encoded {len(voice_samples_base64)} voice samples")
                print()
                
                sample_rate = CHATTERBOX_SAMPLE_RATE  # Use config constant
                load_time = 0.0  # No model loading for RunPod
                
                for i, seg in enumerate(translated_segments):
                    text = seg.get('translated_text', '').strip()
                    
                    if not text:
                        segment_info.append((i, seg['start'], seg['end'], '(skipped)', '(empty)'))
                        continue
                    
                    # Match this segment to a speaker
                    speaker = match_segment_to_speaker(seg['start'], seg['end'])
                    voice_sample_base64 = voice_samples_base64.get(speaker, voice_samples_base64[fallback_speaker])
                    
                    print(f"Segment {i+1}/{total_segments}: {speaker} ({seg['start']:.1f}s - {seg['end']:.1f}s)")
                    
                    try:
                        segment_start = time.time()
                        
                        # Call RunPod endpoint
                        result = run_runpod_sync("chatterbox", {
                            "text": text,
                            "voice_sample_base64": voice_sample_base64,
                            "language": TARGET_LANGUAGE,
                            "cfg_weight": CHATTERBOX_CFG_WEIGHT,
                            "exaggeration": CHATTERBOX_EXAGGERATION,
                        })
                        
                        if result.get("status") == "error":
                            raise RuntimeError(result.get("error", "Unknown error"))
                        
                        # Decode audio from base64
                        audio_bytes = base64.b64decode(result["audio_base64"])
                        
                        # Read as wav
                        import io
                        import soundfile as sf
                        audio_np, sr = sf.read(io.BytesIO(audio_bytes))
                        sample_rate = sr
                        
                        # Normalize
                        max_val = np.max(np.abs(audio_np))
                        if max_val > 0.99:
                            audio_np = audio_np * 0.99 / max_val
                        
                        segment_audios.append((seg['start'], audio_np))
                        segment_info.append((i, seg['start'], seg['end'], speaker, 
                                           text[:30] + '...' if len(text) > 30 else text))
                        
                        segment_time = time.time() - segment_start
                        audio_duration = len(audio_np) / sample_rate
                        print(f"  -> Generated {audio_duration:.1f}s audio in {segment_time:.1f}s")
                        
                    except Exception as e:
                        print(f"  -> FAILED: {str(e)[:50]}")
                        segment_info.append((i, seg['start'], seg['end'], speaker, '(FAILED)'))
                        continue
                
            else:
                # =====================================================
                # Local Chatterbox TTS (Multi-Speaker)
                # =====================================================
                device = "cuda" if torch.cuda.is_available() else "cpu"
                print(f"Loading Chatterbox multilingual model on {device.upper()}...")
                load_start = time.time()
                
                model = ChatterboxMultilingualTTS.from_pretrained(device=device)
                sample_rate = model.sr
                
                load_time = time.time() - load_start
                print(f"  - Model loaded in {load_time:.1f} seconds")
                print()
                
                for i, seg in enumerate(translated_segments):
                    text = seg.get('translated_text', '').strip()
                    
                    if not text:
                        segment_info.append((i, seg['start'], seg['end'], '(skipped)', '(empty)'))
                        continue
                    
                    # Match this segment to a speaker
                    speaker = match_segment_to_speaker(seg['start'], seg['end'])
                    voice_sample = speaker_samples.get(speaker, fallback_sample)
                    
                    print(f"Segment {i+1}/{total_segments}: {speaker} ({seg['start']:.1f}s - {seg['end']:.1f}s)")
                    
                    try:
                        segment_start = time.time()
                        
                        wav = model.generate(
                            text,
                            audio_prompt_path=str(voice_sample),
                            language_id=TARGET_LANGUAGE,
                        )
                        
                        # Convert to numpy
                        audio_np = wav.squeeze().cpu().numpy()
                        
                        # Normalize
                        max_val = np.max(np.abs(audio_np))
                        if max_val > 0.99:
                            audio_np = audio_np * 0.99 / max_val
                        
                        segment_audios.append((seg['start'], audio_np))
                        segment_info.append((i, seg['start'], seg['end'], speaker, 
                                           text[:30] + '...' if len(text) > 30 else text))
                        
                        segment_time = time.time() - segment_start
                        audio_duration = len(audio_np) / sample_rate
                        print(f"  -> Generated {audio_duration:.1f}s audio in {segment_time:.1f}s")
                        
                    except Exception as e:
                        print(f"  -> FAILED: {str(e)[:50]}")
                        segment_info.append((i, seg['start'], seg['end'], speaker, '(FAILED)'))
                        continue
            
            print("-" * 60)
            total_gen_time = time.time() - total_gen_start
            print(f"All segments processed in {total_gen_time:.1f} seconds")
            print()
            
            if not segment_audios:
                print("ERROR: No segments were successfully synthesized!")
            else:
                # Combine all segments with adjusted timing to prevent overlap
                print("Combining segments with natural timing...")
                
                positioned_audios: list[tuple[float, np.ndarray]] = []
                current_end_time = 0.0
                
                for original_start, audio in segment_audios:
                    audio_duration = len(audio) / sample_rate
                    
                    # Start at the later of: original timestamp or when previous segment ends
                    actual_start = max(original_start, current_end_time)
                    positioned_audios.append((actual_start, audio))
                    
                    # Update end time for next segment
                    current_end_time = actual_start + audio_duration
                
                # Output length based on actual audio content
                actual_total_duration = current_end_time
                output_length = int(actual_total_duration * sample_rate) + sample_rate  # +1s buffer
                output_audio = np.zeros(output_length)
                
                # Place audio at calculated positions
                for start_time, audio in positioned_audios:
                    start_sample = int(start_time * sample_rate)
                    end_sample = start_sample + len(audio)
                    
                    if end_sample > output_length:
                        audio = audio[:output_length - start_sample]
                    
                    if start_sample < output_length:
                        output_audio[start_sample:start_sample + len(audio)] += audio
                
                # Normalize to prevent clipping
                max_val = np.max(np.abs(output_audio))
                if max_val > 0.99:
                    output_audio = output_audio * 0.99 / max_val
                
                # Trim trailing silence
                threshold = 0.001
                non_zero_indices = np.where(np.abs(output_audio) > threshold)[0]
                if len(non_zero_indices) > 0:
                    last_sound = non_zero_indices[-1]
                    trim_point = min(last_sound + int(0.5 * sample_rate), len(output_audio))
                    output_audio = output_audio[:trim_point]
                
                # Save the combined audio
                multi_speaker_output = job_dir / "translated_audio_multi.wav"
                write_wav_file(multi_speaker_output, sample_rate, output_audio)
                
                # Store in video_info for later cells
                video_info['multi_speaker_output'] = multi_speaker_output
                video_info['synthesized_audio_path'] = multi_speaker_output
                
                # Get output file info
                output_size = multi_speaker_output.stat().st_size
                output_duration = get_audio_duration(multi_speaker_output)
                
                # Get original video duration for comparison
                original_duration = video_info.get('actual_duration', video_info.get('duration', 0))
                
                print()
                print("Multi-Speaker Synthesis Summary")
                print("-" * 40)
                print(f"Output file: {multi_speaker_output}")
                print(f"File size: {format_file_size(output_size)}")
                print(f"Duration: {format_duration(int(output_duration))} ({output_duration:.2f} seconds)")
                print(f"Original video: {format_duration(int(original_duration))} ({original_duration:.2f} seconds)")
                if output_duration > original_duration:
                    print(f"Note: Audio is {output_duration - original_duration:.1f}s longer than original video")
                print(f"Sample rate: {sample_rate} Hz")
                print(f"Segments synthesized: {len(segment_audios)}/{total_segments}")
                print()
                
                # Timing summary
                print("Timing")
                print("-" * 40)
                if load_time > 0:
                    print(f"Model load: {load_time:.1f} seconds")
                print(f"Total generation: {total_gen_time:.1f} seconds")
                if len(segment_audios) > 0:
                    print(f"Average per segment: {total_gen_time/len(segment_audios):.1f} seconds")
                if output_duration > 0:
                    rtf = total_gen_time / output_duration
                    print(f"Real-time factor: {rtf:.2f}x")
    
        except Exception as e:
            print(f"ERROR: Multi-speaker synthesis failed")
            print(f"  Exception type: {type(e).__name__}")
            print(f"  Details: {str(e)}")
            import traceback
            traceback.print_exc()

print()
print("=" * 60)
if multi_speaker_output and multi_speaker_output.exists():
    print("Multi-speaker synthesis complete! Proceed to audio mixing.")
elif not USE_MULTI_SPEAKER:
    print("Multi-speaker mode skipped. Single-speaker synthesis should be used.")
else:
    print("Multi-speaker synthesis failed. Check errors above.")
print("=" * 60)

## Cell 13: Audio Mixing

This cell mixes the translated speech with the original background audio extracted during the separation step. This creates a natural-sounding output where the translated speech plays over the original music/effects.

### Mixing Process

The mixing uses **ffmpeg's amix filter** to combine two audio tracks:

1. **Speech track** (100% volume): The synthesized translated speech
2. **Background track** (configurable volume): Original background audio from Demucs separation

```
ffmpeg -i speech.wav -i background.wav \
  -filter_complex '[1:a]apad,volume={vol}[bg];[0:a][bg]amix=inputs=2:duration=first:dropout_transition=0' \
  -ac 2 -y mixed_audio.wav
```

### Filter Breakdown

| Component | Description |
|-----------|-------------|
| `[1:a]apad` | Pad background audio with silence if shorter than speech |
| `volume={vol}` | Apply volume adjustment to background (0.0 to 1.0) |
| `[bg]` | Label for processed background stream |
| `amix=inputs=2` | Mix two audio streams together |
| `duration=first` | Output duration matches the speech track (first input) |
| `dropout_transition=0` | No fade when one input ends |
| `-ac 2` | Output stereo audio |

### Configuration

| Variable | Default | Description |
|----------|---------|-------------|
| `BACKGROUND_VOLUME` | 0.3 | Volume level for background (0.0 = muted, 1.0 = full) |

Typical values:
- **0.1 - 0.2**: Background barely audible, speech-focused
- **0.3** (default): Balanced mix, background clearly present
- **0.4 - 0.5**: Background more prominent, cinematic feel
- **0.0**: Background muted (speech only)

### Output

- **mixed_audio.wav**: Final mixed audio with speech and background
- Duration matches the synthesized speech track
- Stereo output format


In [ ]:
# =============================================================================
# Audio Mixing - Mix translated speech with background audio
# =============================================================================

# =============================================================================
# Configuration
# =============================================================================

# Background volume level (0.0 to 1.0)
# - 0.3 (30%) is the default, providing a balanced mix
# - Adjust this to control how prominent the background music/effects are
BACKGROUND_VOLUME = BACKGROUND_VOLUME_DEFAULT  # From config.py (0.3)

# =============================================================================
# Audio Mixing
# =============================================================================

mixed_audio_path = None  # Will hold the path to mixed audio

print("=" * 60)
print("AUDIO MIXING")
print("=" * 60)
print()

# Check prerequisites
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('synthesized_audio_path') is None:
    print("ERROR: synthesized_audio_path not available.")
    print("  Run either the Single-Speaker or Multi-Speaker Synthesis cell first.")
elif video_info.get('background_path') is None:
    print("ERROR: background_path not available. Run the Audio Separation cell first.")
elif not video_info['synthesized_audio_path'].exists():
    print(f"ERROR: Synthesized audio file not found: {video_info['synthesized_audio_path']}")
elif not video_info['background_path'].exists():
    print(f"ERROR: Background audio file not found: {video_info['background_path']}")
else:
    # === Skip check ===
    _skip = False
    _job_dir = video_info["video_path"].parent
    if (_job_dir / "mixed_audio.wav").exists():
        print(f"SKIPPING: Mixed audio already exists")
        mixed_audio_path = _job_dir / "mixed_audio.wav"
        video_info["mixed_audio_path"] = mixed_audio_path
        _skip = True

    if not _skip:
        try:
            speech_path = video_info['synthesized_audio_path']
            background_path = video_info['background_path']
            job_dir = video_info['video_path'].parent
        
            print(f"Speech audio: {speech_path.name}")
            print(f"  Size: {format_file_size(speech_path.stat().st_size)}")
            speech_duration = get_audio_duration(speech_path)
            print(f"  Duration: {format_duration(int(speech_duration))} ({speech_duration:.2f}s)")
            print()
            print(f"Background audio: {background_path.name}")
            print(f"  Size: {format_file_size(background_path.stat().st_size)}")
            background_duration = get_audio_duration(background_path)
            print(f"  Duration: {format_duration(int(background_duration))} ({background_duration:.2f}s)")
            print()
        
            # Validate and clamp background volume
            bg_volume = max(0.0, min(1.0, BACKGROUND_VOLUME))
            if bg_volume != BACKGROUND_VOLUME:
                print(f"WARNING: BACKGROUND_VOLUME clamped from {BACKGROUND_VOLUME} to {bg_volume}")
        
            print(f"Background volume: {bg_volume:.0%}")
            print()
        
            # Output path
            mixed_audio_path = job_dir / "mixed_audio.wav"
        
            print("Mixing audio with ffmpeg...")
            mix_start = time.time()
        
            # Build the ffmpeg filter for mixing
            # [1:a]apad: Pad background with silence if shorter than speech
            # volume={vol}: Apply volume to background
            # amix: Mix the two streams
            # duration=first: Output duration matches the speech (first input)
            filter_complex = (
                f'[1:a]apad,volume={bg_volume}[bg];'
                f'[0:a][bg]amix=inputs=2:duration=first:dropout_transition=0'
            )
        
            result = subprocess.run([
                'ffmpeg',
                '-i', str(speech_path),
                '-i', str(background_path),
                '-filter_complex', filter_complex,
                '-ac', '2',  # Stereo output
                '-y',  # Overwrite output
                str(mixed_audio_path)
            ], capture_output=True, text=True)
        
            mix_time = time.time() - mix_start
        
            if result.returncode != 0:
                print(f"ERROR: ffmpeg mixing failed")
                print(f"  stderr: {result.stderr[:500]}..." if len(result.stderr) > 500 else f"  stderr: {result.stderr}")
                mixed_audio_path = None
            else:
                # Verify output exists
                if not mixed_audio_path.exists():
                    print("ERROR: Output file was not created")
                    mixed_audio_path = None
                else:
                    # Get output info
                    output_size = mixed_audio_path.stat().st_size
                    output_duration = get_audio_duration(mixed_audio_path)
                
                    # Store in video_info for later cells
                    video_info['mixed_audio_path'] = mixed_audio_path
                
                    print()
                    print("Audio Mixing Summary")
                    print("-" * 40)
                    print(f"Output file: {mixed_audio_path}")
                    print(f"File size: {format_file_size(output_size)}")
                    print(f"Duration: {format_duration(int(output_duration))} ({output_duration:.2f} seconds)")
                    print(f"Background volume: {bg_volume:.0%}")
                    print(f"Processing time: {mix_time:.2f} seconds")
                    print()
                
                    # Compare durations
                    if abs(output_duration - speech_duration) > 0.5:
                        print(f"NOTE: Output duration differs from speech by {abs(output_duration - speech_duration):.2f}s")
    
        except subprocess.CalledProcessError as e:
            print(f"ERROR: ffmpeg command failed")
            print(f"  Return code: {e.returncode}")
            if e.stderr:
                print(f"  stderr: {e.stderr[:500]}..." if len(str(e.stderr)) > 500 else f"  stderr: {e.stderr}")
            mixed_audio_path = None
    
        except Exception as e:
            print(f"ERROR: Audio mixing failed")
            print(f"  Exception type: {type(e).__name__}")
            print(f"  Details: {str(e)}")
            mixed_audio_path = None

print()
print("=" * 60)
if mixed_audio_path and mixed_audio_path.exists():
    print(f"Audio mixing complete! Output saved to: {mixed_audio_path}")
    print("Proceed to Subtitle Generation cell.")
else:
    print("Audio mixing failed. Check errors above.")
print("=" * 60)


## Cell 14: Subtitle Generation

This cell generates an **SRT (SubRip Subtitle)** file from the translated segments.

### SRT Format

SRT is a simple text-based subtitle format widely supported by video players:

```
1
00:00:00,000 --> 00:00:05,123
First subtitle line

2
00:00:05,500 --> 00:00:10,000
Second subtitle line
```

Each entry consists of:
1. **Index number** - Sequential counter starting from 1
2. **Timestamp line** - `HH:MM:SS,mmm --> HH:MM:SS,mmm` (start --> end)
3. **Subtitle text** - The actual subtitle content (can span multiple lines)
4. **Blank line** - Separator between entries

### Timestamp Format

| Component | Format | Example | Description |
|-----------|--------|---------|-------------|
| Hours | 00-99 | `01` | Hours (2 digits, zero-padded) |
| Minutes | 00-59 | `30` | Minutes (2 digits, zero-padded) |
| Seconds | 00-59 | `45` | Seconds (2 digits, zero-padded) |
| Milliseconds | 000-999 | `123` | Milliseconds (3 digits, **comma** separator) |

**Important**: SRT uses a **comma** (`,`) before milliseconds, not a period. This is a common source of errors.

### Why Generate SRT?

- **Accessibility** - Enables deaf/hard-of-hearing viewers to follow content
- **SEO** - Search engines can index subtitle text for better discoverability
- **Soft subs** - Can be toggled on/off in video players (vs burned-in text)
- **Easy editing** - Plain text format allows manual corrections
- **Multi-language** - Multiple SRT files can provide subtitles in different languages

### Implementation

The cell uses `format_timestamp_srt()` (from Cell 1 setup) to convert float seconds to SRT format,
then iterates through `translated_segments` to build the subtitle file.


In [ ]:
# =============================================================================
# Subtitle Generation - Create SRT file from translated segments
# =============================================================================

subtitle_path = None  # Will hold the path to generated SRT file

print("=" * 60)
print("SUBTITLE GENERATION")
print("=" * 60)
print()

# Check prerequisites
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('translated_segments') is None:
    print("ERROR: translated_segments not available. Run the Translation cell first.")
elif len(video_info.get('translated_segments', [])) == 0:
    print("ERROR: No translated segments found.")
else:
    # === Skip check ===
    _skip = False
    _job_dir = video_info["video_path"].parent
    if (_job_dir / "subtitles.srt").exists():
        print(f"SKIPPING: Subtitles already exist")
        subtitle_path = _job_dir / "subtitles.srt"
        video_info["subtitle_path"] = subtitle_path
        _skip = True

    if not _skip:
        try:
            translated_segments = video_info['translated_segments']
            job_dir = video_info['video_path'].parent
        
            print(f"Total segments to convert: {len(translated_segments)}")
            print()
        
            # Build SRT content
            srt_lines: list[str] = []
            subtitle_index = 1
            skipped_count = 0
        
            for seg in translated_segments:
                text = seg.get('translated_text', '').strip()
            
                # Skip empty segments
                if not text:
                    skipped_count += 1
                    continue
            
                start_time = seg.get('start', 0.0)
                end_time = seg.get('end', 0.0)
            
                # Add SRT entry
                srt_lines.append(str(subtitle_index))
                srt_lines.append(f"{format_timestamp_srt(start_time)} --> {format_timestamp_srt(end_time)}")
                srt_lines.append(text)
                srt_lines.append("")  # Blank line between entries
            
                subtitle_index += 1
        
            # Calculate total subtitle count (excluding skipped)
            total_subtitles = subtitle_index - 1
        
            print(f"Subtitles generated: {total_subtitles}")
            if skipped_count > 0:
                print(f"Empty segments skipped: {skipped_count}")
            print()
        
            # Write SRT file
            subtitle_path = job_dir / "subtitles.srt"
            srt_content = "\n".join(srt_lines)
        
            with open(subtitle_path, 'w', encoding='utf-8') as f:
                f.write(srt_content)
        
            # Store in video_info for later cells
            video_info['subtitle_path'] = subtitle_path
        
            print("Subtitle Generation Summary")
            print("-" * 40)
            print(f"Output file: {subtitle_path}")
            print(f"File size: {format_file_size(subtitle_path.stat().st_size)}")
            print(f"Total subtitles: {total_subtitles}")
            print()
        
            # Display first 10 subtitle entries as preview
            print("Subtitle Preview (first 10 entries)")
            print("-" * 40)
        
            # Re-iterate to display preview (up to 10 entries)
            preview_count = 0
            for seg in translated_segments:
                text = seg.get('translated_text', '').strip()
                if not text:
                    continue
            
                preview_count += 1
                if preview_count > 10:
                    remaining = total_subtitles - 10
                    print(f"\n... and {remaining} more subtitle(s)")
                    break
            
                start_time = seg.get('start', 0.0)
                end_time = seg.get('end', 0.0)
            
                print(f"{preview_count}")
                print(f"{format_timestamp_srt(start_time)} --> {format_timestamp_srt(end_time)}")
                # Truncate long text for display
                display_text = text[:80] + "..." if len(text) > 80 else text
                print(f"{display_text}")
                print()
    
        except IOError as e:
            print(f"ERROR: Failed to write SRT file")
            print(f"  Exception type: {type(e).__name__}")
            print(f"  Details: {str(e)}")
            subtitle_path = None
    
        except Exception as e:
            print(f"ERROR: Subtitle generation failed")
            print(f"  Exception type: {type(e).__name__}")
            print(f"  Details: {str(e)}")
            subtitle_path = None

print()
print("=" * 60)
if subtitle_path and subtitle_path.exists():
    print(f"Subtitle generation complete! Output saved to: {subtitle_path}")
    print("Proceed to Final Video Merge cell.")
else:
    print("Subtitle generation failed. Check errors above.")
print("=" * 60)


## Cell 14: Lip Sync Processing (MuseTalk)

This cell applies AI lip synchronization using **MuseTalk** to make the video's lip movements match the translated audio.

### Prerequisites

MuseTalk requires a separate conda environment. Setup once:

```bash
# Create musetalk environment
conda create -n musetalk python=3.10
conda activate musetalk

# Install dependencies
pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
pip install -r MuseTalk/requirements.txt
pip install --no-cache-dir -U openmim
mim install mmengine mmcv==2.0.1 mmdet==3.1.0 mmpose==1.1.0
```

### What This Cell Does

1. **Loads mixed audio** (translated speech + background)
2. **Processes video** with MuseTalk
3. **Generates lip movements** matching the new audio
4. **Saves lip-synced video** for final merge

### Model Details

| Property | Value |
|----------|-------|
| Model | MuseTalk (latent space inpainting) |
| Face Region | 256x256 pixels |
| Languages | Multi-language support |
| VRAM | ~8GB recommended |

### Expected Output

- `lip_synced.mp4` - Video with lip movements matching translated audio

In [ ]:
# =============================================================================
# Lip Sync Processing - Apply MuseTalk to match video lips to audio
# =============================================================================

if not USE_RUNPOD_ENDPOINTS:
    from lip_sync import apply_lip_sync, check_musetalk_setup

lip_synced_path = None  # Will hold the path to lip-synced video

print("=" * 60)
print(f"LIP SYNC PROCESSING ({'RunPod MuseTalk' if USE_RUNPOD_ENDPOINTS else 'Local MuseTalk'})")
print("=" * 60)
print()

# Check prerequisites
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('video_path') is None:
    print("ERROR: video_path not available. Run the Video Download cell first.")
elif video_info.get('mixed_audio_path') is None:
    print("ERROR: mixed_audio_path not available. Run the Audio Mixing cell first.")
else:
    # Check MuseTalk setup for local mode
    if not USE_RUNPOD_ENDPOINTS:
        print("Checking MuseTalk setup...")
        status = check_musetalk_setup()
        
        if status['issues']:
            print("MuseTalk setup issues found:")
            for issue in status['issues']:
                print(f"  - {issue}")
            print()
            print("Please fix the issues above before running lip sync.")
            print("Lip sync will be skipped - final merge will use original video.")
        else:
            print("MuseTalk setup OK")
            print()
    
    # === Skip check ===
    _skip = False
    _job_dir = video_info["video_path"].parent
    if (_job_dir / "lip_synced.mp4").exists():
        print(f"SKIPPING: Lip-synced video already exists")
        lip_synced_path = _job_dir / "lip_synced.mp4"
        video_info["lip_synced_path"] = lip_synced_path
        _skip = True

    if not _skip and (USE_RUNPOD_ENDPOINTS or not status.get('issues')):
        try:
            video_path = video_info['video_path']
            audio_path = video_info['mixed_audio_path']
            job_dir = video_path.parent
            output_path = job_dir / "lip_synced.mp4"
            
            print(f"Input video: {video_path.name}")
            print(f"Input audio: {audio_path.name}")
            print(f"Output: {output_path.name}")
            print()
            
            print("Processing with MuseTalk...")
            print("(This may take several minutes depending on video length)")
            print("-" * 40)
            
            sync_start = time.time()
            
            if USE_RUNPOD_ENDPOINTS:
                # =====================================================
                # RunPod MuseTalk Endpoint
                # =====================================================
                print("Using RunPod MuseTalk endpoint...")
                print()
                
                # Read video and audio, encode to base64
                with open(video_path, "rb") as f:
                    video_base64 = base64.b64encode(f.read()).decode("utf-8")
                print(f"  Video encoded: {len(video_base64) / 1e6:.1f} MB base64")
                
                with open(audio_path, "rb") as f:
                    audio_base64 = base64.b64encode(f.read()).decode("utf-8")
                print(f"  Audio encoded: {len(audio_base64) / 1e6:.1f} MB base64")
                print()
                
                # Call RunPod endpoint with longer timeout for lip sync
                result = run_runpod_sync("musetalk", {
                    "video_base64": video_base64,
                    "audio_base64": audio_base64,
                    "use_float16": True,
                }, timeout=1800.0)  # 30 minute timeout
                
                if result.get("status") == "error":
                    raise RuntimeError(f"MuseTalk failed: {result.get('error', 'Unknown error')}")
                
                # Download video from RunPod storage URL
                print("  Downloading lip-synced video from RunPod storage...")
                output_response = httpx.get(result["video_url"], timeout=600.0)
                output_data = output_response.content
                
                with open(output_path, "wb") as f:
                    f.write(output_data)
                print(f"  Output saved: {len(output_data) / 1e6:.1f} MB")
                
                # Get metrics
                metrics = result.get("metrics", {})
                print(f"  Inference time: {metrics.get('inference_seconds', 'N/A')}s")
                
                lip_synced_path = output_path
                
            else:
                # =====================================================
                # Local MuseTalk
                # =====================================================
                lip_synced_path = apply_lip_sync(
                    video_path=video_path,
                    audio_path=audio_path,
                    output_path=output_path,
                    use_float16=True,  # Use FP16 for faster processing
                )
            
            sync_time = time.time() - sync_start
            
            print("-" * 40)
            print()
            
            # Store in video_info
            video_info['lip_synced_path'] = lip_synced_path
            
            # Get output info
            output_size = lip_synced_path.stat().st_size
            output_duration = get_audio_duration(lip_synced_path)
            
            print("Lip Sync Summary")
            print("-" * 40)
            print(f"Output file: {lip_synced_path}")
            print(f"File size: {format_file_size(output_size)}")
            print(f"Duration: {format_duration(int(output_duration))} ({output_duration:.2f}s)")
            print(f"Processing time: {sync_time:.1f} seconds")
            
        except Exception as e:
            print(f"ERROR: Lip sync failed")
            print(f"  Exception type: {type(e).__name__}")
            print(f"  Details: {str(e)}")
            import traceback
            traceback.print_exc()
            print()
            print("Lip sync will be skipped - final merge will use original video.")

print()
print("=" * 60)
if lip_synced_path and lip_synced_path.exists():
    print("Lip sync complete! Proceed to subtitle generation or final merge.")
else:
    print("Lip sync skipped. Final merge will use original video.")
print("=" * 60)


## Cell 15: Final Video Merge

This cell merges the translated audio with the original video and embeds subtitles as soft subs.

### Merge Process

The merge uses **ffmpeg** to combine three streams:

1. **Video stream** (from original video): Copied without re-encoding
2. **Audio stream** (from mixed audio): Encoded as AAC
3. **Subtitle stream** (from SRT file): Embedded as soft subs

```bash
ffmpeg -i video.mp4 -i mixed_audio.wav -i subtitles.srt \
  -c:v copy -c:a aac -c:s mov_text \
  -map 0:v:0 -map 1:a:0 -map 2:0 \
  -metadata:s:s:0 language=spa \
  -shortest -y output.mp4
```

### FFmpeg Options Explained

| Option | Description |
|--------|-------------|
| `-c:v copy` | Copy video stream without re-encoding (fast, lossless) |
| `-c:a aac` | Encode audio as AAC (standard MP4 audio codec) |
| `-c:s mov_text` | Use mov_text codec for MP4-compatible soft subtitles |
| `-map 0:v:0` | Map first video stream from first input (video file) |
| `-map 1:a:0` | Map first audio stream from second input (audio file) |
| `-map 2:0` | Map first stream from third input (subtitle file) |
| `-metadata:s:s:0 language=X` | Set subtitle language metadata (ISO 639-2 code) |
| `-shortest` | Output duration matches shortest input stream |
| `-y` | Overwrite output file without asking |

### Soft Subtitles vs Hard Subtitles

| Type | Description | Pros | Cons |
|------|-------------|------|------|
| **Soft subs** (mov_text) | Embedded as separate stream | Toggleable, editable | May not work in all players |
| **Hard subs** (burned-in) | Rendered into video frames | Always visible | Cannot turn off, requires re-encode |

We use soft subs for flexibility - viewers can toggle them on/off in their video player.

### Output

- **{title}_translated.mp4**: Final video with translated audio and embedded subtitles
- File size similar to original (video stream is copied)
- Total pipeline execution time displayed


In [ ]:
# =============================================================================
# Final Video Merge - Merge video, translated audio, and subtitles
# =============================================================================

final_output_path = None  # Will hold the path to final video

print("=" * 60)
print("FINAL VIDEO MERGE")
print("=" * 60)
print()

# Check prerequisites
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('video_path') is None:
    print("ERROR: video_path not available. Run the Video Download cell first.")
elif video_info.get('mixed_audio_path') is None:
    print("ERROR: mixed_audio_path not available. Run the Audio Mixing cell first.")
elif not video_info['video_path'].exists():
    print(f"ERROR: Video file not found: {video_info['video_path']}")
elif not video_info['mixed_audio_path'].exists():
    print(f"ERROR: Mixed audio file not found: {video_info['mixed_audio_path']}")
else:
    try:
        # Use lip-synced video if available, otherwise original
        lip_synced = video_info.get('lip_synced_path')
        if lip_synced and lip_synced.exists():
            video_path = lip_synced
            print("Using lip-synced video for final merge")
        else:
            video_path = video_info['video_path']
            print("Using original video (lip sync not available)")
        audio_path = video_info['mixed_audio_path']
        subtitle_path = video_info.get('subtitle_path')  # May be None
        job_dir = video_path.parent
        
        # Get video title for output filename
        title = video_info.get('title', 'video')
        # Sanitize title for filename (keep alphanumeric, spaces, hyphens, underscores)
        safe_title = "".join(c for c in title if c.isalnum() or c in " -_")[:40]
        if not safe_title:
            safe_title = "translated_video"
        
        # Output path
        final_output_path = job_dir / f"{safe_title}_translated.mp4"
        
        print("Input Files")
        print("-" * 40)
        print(f"Video: {video_path.name}")
        print(f"  Size: {format_file_size(video_path.stat().st_size)}")
        print(f"Audio: {audio_path.name}")
        print(f"  Size: {format_file_size(audio_path.stat().st_size)}")
        
        # Check if subtitles are available
        has_subtitles = subtitle_path is not None and subtitle_path.exists()
        if has_subtitles:
            print(f"Subtitles: {subtitle_path.name}")
            print(f"  Size: {format_file_size(subtitle_path.stat().st_size)}")
        else:
            print("Subtitles: None (will merge without subtitles)")
        print()
        
        # Get subtitle language code (ISO 639-2 for ffmpeg metadata)
        subtitle_lang = get_iso_639_2_code(TARGET_LANGUAGE)
        print(f"Subtitle language: {subtitle_lang} (from {TARGET_LANGUAGE})")
        print(f"Output file: {final_output_path.name}")
        print()
        
        print("Merging with ffmpeg...")
        merge_start = time.time()
        
        # Build ffmpeg command based on whether subtitles are provided
        if has_subtitles:
            # With subtitles: include subtitle input and mapping
            cmd = [
                'ffmpeg',
                '-i', str(video_path),
                '-i', str(audio_path),
                '-i', str(subtitle_path),
                '-c:v', 'copy',           # Copy video stream (no re-encoding)
                '-c:a', 'aac',            # Encode audio as AAC
                '-c:s', 'mov_text',       # Soft subtitles codec for MP4
                '-map', '0:v:0',          # Map video from first input
                '-map', '1:a:0',          # Map audio from second input
                '-map', '2:0',            # Map subtitles from third input
                '-metadata:s:s:0', f'language={subtitle_lang}',  # Set subtitle language
                              # Output duration matches shortest input
                '-y',                     # Overwrite without asking
                str(final_output_path)
            ]
        else:
            # Without subtitles: simpler merge
            cmd = [
                'ffmpeg',
                '-i', str(video_path),
                '-i', str(audio_path),
                '-c:v', 'copy',           # Copy video stream (no re-encoding)
                '-c:a', 'aac',            # Encode audio as AAC
                '-map', '0:v:0',          # Map video from first input
                '-map', '1:a:0',          # Map audio from second input
                              # Output duration matches shortest input
                '-y',                     # Overwrite without asking
                str(final_output_path)
            ]
        
        # Run ffmpeg
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        merge_time = time.time() - merge_start
        
        if result.returncode != 0:
            print(f"ERROR: ffmpeg failed with return code {result.returncode}")
            print(f"stderr: {result.stderr}")
        else:
            # Store in video_info for reference
            video_info['final_output_path'] = final_output_path
            
            # Get output file info
            output_size = final_output_path.stat().st_size
            output_duration = get_audio_duration(final_output_path)  # Works for video too
            
            # Calculate total pipeline time if pipeline_start_time exists
            total_pipeline_time = None
            if 'pipeline_start_time' in video_info:
                total_pipeline_time = time.time() - video_info['pipeline_start_time']
            
            print()
            print("Final Video Merge Summary")
            print("=" * 40)
            print(f"Output file: {final_output_path}")
            print(f"File size: {format_file_size(output_size)}")
            print(f"Duration: {format_duration(int(output_duration))} ({output_duration:.2f}s)")
            print(f"Subtitles embedded: {'Yes' if has_subtitles else 'No'}")
            print()
            print("Timing")
            print("-" * 40)
            print(f"Merge time: {merge_time:.2f}s")
            if total_pipeline_time is not None:
                print(f"Total pipeline time: {format_duration(int(total_pipeline_time))} ({total_pipeline_time:.1f}s)")
            print()
            print("SUCCESS: Final video created!")
            print(f"  {final_output_path}")
            
    except subprocess.CalledProcessError as e:
        print(f"ERROR: ffmpeg command failed: {e}")
        if e.stderr:
            print(f"stderr: {e.stderr}")
    except Exception as e:
        print(f"ERROR: {type(e).__name__}: {e}")


## Cell 16: Pipeline Summary and Quality Checklist

This final cell provides a comprehensive summary of the translation pipeline execution:

### Summary Includes

| Section | Description |
|---------|-------------|
| **Pipeline Stages** | Status of each processing stage (completed/skipped) |
| **Output Files** | Paths to all generated files with sizes |
| **Quality Checklist** | Manual verification items for quality assurance |
| **Debugging Tips** | Common issues and how to investigate them |

### Pipeline Stages

The translation pipeline consists of 14 main stages:

1. **Setup & Config** - Load dependencies and API clients
2. **Input Config** - Set video URL, language, options
3. **Video Info** - Fetch metadata without downloading
4. **Video Download** - Download and optionally trim video
5. **Audio Extraction** - Extract 16kHz mono WAV for Whisper
6. **Audio Separation** - Split vocals from background (Demucs)
7. **Speaker Diarization** - Identify who speaks when
8. **Voice Sample Extraction** - Extract samples for voice cloning
9. **Transcription** - Convert speech to text (Whisper)
10. **Translation** - Translate segments (LLM)
11. **Speech Synthesis** - Generate translated speech (Chatterbox)
12. **Audio Mixing** - Combine speech with background
13. **Subtitle Generation** - Create SRT file
14. **Video Merge** - Produce final translated video

### Quality Verification

After running the pipeline, manually verify:

- [ ] **Transcription accuracy** - Play vocals.wav, compare to transcript
- [ ] **Translation quality** - Check translated segments make sense in context
- [ ] **Speaker matching** - Verify correct voices for each speaker
- [ ] **Audio timing** - Speech should align with original video action
- [ ] **Subtitle sync** - Subtitles appear at correct times
- [ ] **Mix balance** - Background music not too loud/quiet
- [ ] **Final video** - Complete playthrough with no artifacts

### Debugging Approaches

| Issue | How to Debug |
|-------|-------------|
| Poor transcription | Check vocals.wav quality, try original audio instead |
| Wrong translations | Review LLM prompts, try smaller batches |
| Voice mismatch | Verify speaker_voice_urls mapping, check sample quality |
| Timing drift | Compare segment durations, check synthesis speed settings |
| Subtitle desync | Verify translated_segments timestamps match synthesis |
| Background too loud | Adjust BACKGROUND_VOLUME (default 0.3) |
| Output corrupted | Check ffmpeg stderr, verify input file integrity |

In [ ]:
# =============================================================================
# Pipeline Summary and Quality Checklist
# =============================================================================

print("=" * 70)
print("PIPELINE SUMMARY AND QUALITY CHECKLIST")
print("=" * 70)
print()

# Check if video_info exists
if video_info is None:
    print("ERROR: video_info not available. Run earlier cells first.")
else:
    # -------------------------------------------------------------------------
    # Pipeline Stage Status
    # -------------------------------------------------------------------------
    print("PIPELINE STAGE STATUS")
    print("-" * 70)
    
    stages = [
        ("1. Video Info", 'title', "Video metadata retrieved"),
        ("2. Video Download", 'video_path', "Video downloaded"),
        ("3. Audio Extraction", 'audio_path', "Audio extracted (16kHz mono WAV)"),
        ("4. Audio Separation", 'vocals_path', "Vocals separated (Demucs)"),
        ("5. Speaker Diarization", 'diarization_segments', "Speakers identified"),
        ("6. Voice Samples", 'speaker_voice_urls', "Voice samples extracted"),
        ("7. Transcription", 'transcription_segments', "Speech transcribed (Whisper)"),
        ("8. Translation", 'translated_segments', "Text translated (LLM)"),
        ("9. Speech Synthesis", 'synthesized_audio_path', "Speech synthesized (Chatterbox)"),
        ("10. Audio Mixing", 'mixed_audio_path', "Audio mixed with background"),
        ("11. Subtitles", 'subtitle_path', "SRT subtitles generated"),
        ("12. Final Video", 'final_output_path', "Video merged and complete"),
    ]
    
    completed_count = 0
    for stage_name, key, description in stages:
        value = video_info.get(key)
        if value is not None:
            # Check if it's a Path and exists, or just exists as data
            if hasattr(value, 'exists'):
                status = "✓ DONE" if value.exists() else "✗ FILE MISSING"
            else:
                status = "✓ DONE"
            completed_count += 1
        else:
            status = "- SKIPPED"
        print(f"{stage_name:25} {status:15} {description}")
    
    print()
    print(f"Completed: {completed_count}/{len(stages)} stages")
    print()
    
    # -------------------------------------------------------------------------
    # Output Files
    # -------------------------------------------------------------------------
    print("OUTPUT FILES")
    print("-" * 70)
    
    output_files = [
        ('video_path', 'Original video'),
        ('audio_path', 'Extracted audio (16kHz WAV)'),
        ('vocals_path', 'Vocals track'),
        ('background_path', 'Background track'),
        ('synthesized_audio_path', 'Synthesized speech'),
        ('mixed_audio_path', 'Mixed audio (speech + background)'),
        ('subtitle_path', 'Subtitles (SRT)'),
        ('final_output_path', 'FINAL OUTPUT VIDEO'),
    ]
    
    total_size = 0
    for key, description in output_files:
        path = video_info.get(key)
        if path is not None and hasattr(path, 'exists') and path.exists():
            size = path.stat().st_size
            total_size += size
            size_str = format_file_size(size)
            print(f"{description:35} {size_str:>12}   {path}")
        elif path is not None:
            print(f"{description:35} {'(missing)':>12}   {path}")
    
    print("-" * 70)
    print(f"{'Total size':35} {format_file_size(total_size):>12}")
    print()
    
    # -------------------------------------------------------------------------
    # Voice Samples (if multi-speaker)
    # -------------------------------------------------------------------------
    speaker_samples = video_info.get('speaker_samples', {})
    if speaker_samples:
        print("VOICE SAMPLES")
        print("-" * 70)
        for speaker_id, sample_path in speaker_samples.items():
            if sample_path.exists():
                size = format_file_size(sample_path.stat().st_size)
                print(f"{speaker_id:20} {size:>12}   {sample_path}")
        print()
    
    # -------------------------------------------------------------------------
    # Pipeline Statistics
    # -------------------------------------------------------------------------
    print("PIPELINE STATISTICS")
    print("-" * 70)
    
    # Video/audio info
    if video_info.get('title'):
        print(f"Video title: {video_info['title']}")
    if video_info.get('actual_duration'):
        print(f"Video duration: {format_duration(int(video_info['actual_duration']))} ({video_info['actual_duration']:.2f}s)")
    
    # Speaker info
    num_speakers = video_info.get('num_speakers', 0)
    print(f"Speakers detected: {num_speakers}")
    
    # Segment counts
    transcription_segments = video_info.get('transcription_segments', [])
    translated_segments = video_info.get('translated_segments', [])
    print(f"Transcription segments: {len(transcription_segments)}")
    print(f"Translated segments: {len(translated_segments)}")
    
    # Synthesis mode
    if video_info.get('single_speaker_output'):
        print(f"Synthesis mode: Single-speaker")
    elif video_info.get('multi_speaker_output'):
        print(f"Synthesis mode: Multi-speaker")
    
    # Language info
    detected_lang = video_info.get('detected_language', 'unknown')
    print(f"Detected language: {detected_lang}")
    print(f"Target language: {TARGET_LANGUAGE} ({get_language_name(TARGET_LANGUAGE)})")
    print()
    
    # -------------------------------------------------------------------------
    # Total Pipeline Time
    # -------------------------------------------------------------------------
    if video_info.get('pipeline_start_time'):
        total_time = time.time() - video_info['pipeline_start_time']
        print("TOTAL EXECUTION TIME")
        print("-" * 70)
        print(f"Pipeline duration: {format_duration(int(total_time))} ({total_time:.1f}s)")
        
        # Calculate real-time ratio
        if video_info.get('actual_duration'):
            ratio = total_time / video_info['actual_duration']
            print(f"Processing ratio: {ratio:.2f}x real-time")
        print()
    
    # -------------------------------------------------------------------------
    # Quality Checklist
    # -------------------------------------------------------------------------
    print("QUALITY CHECKLIST")
    print("-" * 70)
    print("Manual verification items (mark as you check):")
    print()
    
    checklist = [
        "[ ] Transcription accuracy - Compare vocals.wav playback to transcript",
        "[ ] Translation quality - Verify translated text makes contextual sense",
        "[ ] Speaker voice matching - Correct voice used for each speaker",
        "[ ] Audio timing alignment - Speech syncs with video action",
        "[ ] Subtitle synchronization - Text appears at correct times",
        "[ ] Mix balance - Background volume appropriate (adjust BACKGROUND_VOLUME if needed)",
        "[ ] Final video playthrough - No audio/video artifacts",
    ]
    
    for item in checklist:
        print(f"  {item}")
    print()
    
    # -------------------------------------------------------------------------
    # Quick Debug Commands
    # -------------------------------------------------------------------------
    print("QUICK DEBUG COMMANDS")
    print("-" * 70)
    print("Copy-paste these commands in new cells to debug specific issues:")
    print()
    print("# Play vocals audio (check transcription source quality):")
    print(f"# !ffplay -autoexit {{video_info.get('vocals_path', 'N/A')}}")
    print()
    print("# Play synthesized audio (check TTS output):")
    print(f"# !ffplay -autoexit {{video_info.get('synthesized_audio_path', 'N/A')}}")
    print()
    print("# Compare original vs translated segment (replace N with segment index):")
    print("# N = 0")
    print("# orig = video_info['transcription_segments'][N]")
    print("# trans = video_info['translated_segments'][N]")
    print("# print(f\"Original: {orig['text']}\\nTranslated: {trans['translated_text']}\")")
    print()
    print("# View diarization segment-to-speaker mapping:")
    print("# for seg in video_info.get('diarization_segments', [])[:10]:")
    print("#     print(f\"{seg['speaker']}: {seg['start']:.2f}s - {seg['stop']:.2f}s\")")
    print()
    
    # -------------------------------------------------------------------------
    # Final Output Path (prominent display)
    # -------------------------------------------------------------------------
    final_path = video_info.get('final_output_path')
    if final_path and final_path.exists():
        print("=" * 70)
        print("FINAL OUTPUT")
        print("=" * 70)
        print(f"\n  >>> {final_path} <<<\n")
        print(f"  Size: {format_file_size(final_path.stat().st_size)}")
        print(f"  Duration: {format_duration(int(get_audio_duration(final_path)))}")
        print()
    else:
        print("=" * 70)
        print("NOTE: Final video not yet created. Run the Final Video Merge cell.")
        print("=" * 70)